# DS5 Step 2 — Phenotyping and Behavioral Subgroup Discovery

**Notebook:** `100_phenotype_exploration.ipynb`  
**Stage:** Stage 4, Step 2  
**Dataset:** DS5 — German adolescents RCT, N=573  
**Date:** 2026-04-18  
**Blueprint:** `outputs/reports/stage4/ds5_1st_phenotyping_blueprint.md`

---

## A01 — What this notebook is and is not

**Step 2 (this notebook):** Use completed Step 1 model parameters as features for behavioral
subgroup discovery. Describe who differs from whom and along which dimensions.

**NOT Step 3:** No intervention fitting, no optimal friction prescription per user,
no personalization logic. The moment this notebook predicts which friction type a user
should receive, it has crossed into Step 3 territory. Avoid this.

**Proxy warning (mandatory):** All features derive from X_t = n_opened / n_events — a
count-based behavioral proxy. X_t approximates override tendency. It does NOT directly
measure urge, self-control capacity, habit strength, or any latent psychological state.
All cluster labels and phenotype descriptions inherit this limitation.


## A02 — FLAG system for Step 2

| Flag | Decision applied | Alternative deferred |
|---|---|---|
| FLAG P1 | **Core clustering: 5 features** (sig_hat, H_emp, x_mean, HHI_net, rho_self) | H_out as 6th feature (condition-adjusted) |
| FLAG P2 | **K-means** as primary algorithm | GMM for soft membership (run as comparison) |
| FLAG P3 | **k selected by silhouette + dendrogram alignment** | BIC/AIC from GMM |
| FLAG P4 | **Psychosocial scales post-hoc overlay only** | Supervised clustering (Step 3 territory) |
| FLAG P5 | **No imputation**; participants missing any core feature excluded from Phase 1 | Multiple imputation (deferred) |
| FLAG P6 | **H_out excluded from Phase 1** (condition confound) | Residualize H_out on condition first |

**Minimal Phase 1 core set (FLAG P1):**
```
PHASE1_CORE = ['sig_hat', 'H_emp', 'x_mean', 'HHI_net', 'rho_self']
```

**Expanded sensitivity set (Phase 2 — explicitly separate from Phase 1):**
```
PHASE2_SENSITIVITY = PHASE1_CORE + ['H_hour', 'H_app', 'chi_behav', 'H_out_resid']
```
Phase 2 results are compared with Phase 1 using the Adjusted Rand Index.
They are NOT a replacement for Phase 1 — they are a sensitivity check.


In [ ]:
# C01 — Imports and configuration
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, calinski_harabasz_score, davies_bouldin_score,
    adjusted_rand_score
)
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from scipy import stats

# ── Paths ──────────────────────────────────────────────────
ROOT        = Path('..')
MODEL_ROOT  = ROOT / 'outputs' / 'model_outputs'
FEAT_ROOT   = ROOT / 'outputs' / 'features' / 'ds5'
OUTPUT_DIR  = MODEL_ROOT / 'phenotyping'
FIG_DIR     = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)
FIG_DIR.mkdir(exist_ok=True)

RANDOM_STATE = 42
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})

# ── Explicit feature set declarations (FLAG P1 boundary) ───
PHASE1_CORE = ['sig_hat', 'H_emp', 'x_mean', 'HHI_net', 'rho_self']
# Phase 2 extended set — deliberately kept separate
PHASE2_SENSITIVITY_EXTRA = ['H_hour', 'H_app', 'chi_behav']  # H_out_resid added after residualization
PHASE2_EXTENDED = PHASE1_CORE + PHASE2_SENSITIVITY_EXTRA

PSYCHOSOCIAL_COLS = [
    'fomo_mean_pre', 'belong_mean_pre', 'ostracism_mean_pre',
    'digital_wellbeing_pre', 'social_media_wellbeing_pre'
]

print('Imports complete.')
print(f'Phase 1 core features: {PHASE1_CORE}')
print(f'Phase 2 sensitivity extras: {PHASE2_SENSITIVITY_EXTRA}')


## B — Data Assembly


In [ ]:
# B01-B05 — Load all 5 model CSVs and survey features

# ── Load ──────────────────────────────────────────────────
int_raw  = pd.read_csv(MODEL_ROOT / 'information_theory'    / 'participant_entropy_metrics.csv')
stop_raw = pd.read_csv(MODEL_ROOT / 'stochastic_processes'  / 'participant_parameters.csv')
dys_raw  = pd.read_csv(MODEL_ROOT / 'dynamical_systems'     / 'participant_dynamics_parameters.csv')
stm_raw  = pd.read_csv(MODEL_ROOT / 'statistical_mechanics' / 'participant_stm_parameters.csv')
net_raw  = pd.read_csv(MODEL_ROOT / 'network_models'        / 'participant_network_metrics.csv')
surv_raw = pd.read_csv(FEAT_ROOT  / 'ds5_survey_features.csv')

print(f'InT:    {len(int_raw):4d} rows')
print(f'StoP:   {len(stop_raw):4d} rows')
print(f'DyS:    {len(dys_raw):4d} rows')
print(f'StM:    {len(stm_raw):4d} rows')
print(f'NeT:    {len(net_raw):4d} rows')
print(f'Survey: {len(surv_raw):4d} rows')


In [ ]:
# B03 — Select non-redundant columns from each model output
#        Rationale documented in blueprint §3.2

# InT: source of H_emp, H_out, H_app, H_hour, H_SDE; also provides study_condition
#      Drop sig_hat here (taken from StoP) to avoid merge collision
int_keep = ['participant_id_canonical', 'study_condition',
             'H_out', 'H_app', 'H_hour', 'H_emp', 'H_SDE']

# StoP: primary source of sig_hat, mu_hat, x_mean, sigma_mu_ratio
stop_keep = ['participant_id_canonical', 'n_days',
              'x_mean', 'mu_hat', 'sig_hat', 'sigma_mu_ratio']

# DyS: regime labels and deviation metric only (no redundant continuous cols)
dys_keep = ['participant_id_canonical',
             'is_dys_eligible', 'regime_label', 'predicted_boundary_fp',
             'deviation_from_predicted_fixed_point', 'behaviorally_deterministic']

# StM: thermodynamic analogs; exclude monotone transforms of sig_hat
stm_keep = ['participant_id_canonical',
             'beta_behav', 'T_behav', 'p_use_approx', 'chi_behav',
             'F_behav', 'H_StM_participant', 'four_model_convergent',
             'behavioral_regime', 'high_chi']

# NeT: all metrics (will mask non-eligible rows below)
net_keep = ['participant_id_canonical',
             'is_net_eligible', 'is_stationary_eligible',
             'HHI_net', 'top1_share_net', 'top3_share_net',
             'rho_self', 'H_trans', 'H_pi', 'I_consec']

# Survey: psychosocial scales pre-intervention
surv_keep = ['participant_id_canonical',
              'fomo_mean_pre', 'belong_mean_pre', 'ostracism_mean_pre',
              'digital_wellbeing_pre', 'social_media_wellbeing_pre']

# ── Merge ─────────────────────────────────────────────────
df = int_raw[int_keep].copy()
df = df.merge(stop_raw[stop_keep], on='participant_id_canonical', how='left')
df = df.merge(dys_raw[dys_keep],   on='participant_id_canonical', how='left')
df = df.merge(stm_raw[stm_keep],   on='participant_id_canonical', how='left')

# NeT: mask NeT features for non-eligible participants (< 20 transitions)
net_sub = net_raw[net_keep].copy()
net_feat_cols = ['HHI_net', 'top1_share_net', 'top3_share_net', 'rho_self', 'H_trans', 'I_consec']
not_eligible = ~net_sub['is_net_eligible'].astype(bool)
for col in net_feat_cols:
    net_sub.loc[not_eligible, col] = np.nan
not_stat = ~net_sub['is_stationary_eligible'].astype(bool)
net_sub.loc[not_stat, 'H_pi'] = np.nan

df = df.merge(net_sub, on='participant_id_canonical', how='left')
df = df.merge(surv_raw[surv_keep], on='participant_id_canonical', how='left')

print(f'Merged dataframe: {df.shape[0]} rows x {df.shape[1]} columns')
print(f'\nCondition distribution:')
print(df['study_condition'].value_counts(dropna=False).to_string())


In [ ]:
# B06 — Population counts for all core and secondary features

all_candidate = PHASE1_CORE + PHASE2_SENSITIVITY_EXTRA + PSYCHOSOCIAL_COLS + [
    'mu_hat', 'sigma_mu_ratio', 'chi_behav', 'H_out',
    'is_dys_eligible', 'behaviorally_deterministic', 'four_model_convergent'
]

print('Feature completeness (N valid / 572 total):')
print('-' * 50)
for col in all_candidate:
    if col in df.columns:
        n_valid = df[col].notna().sum()
        print(f'  {col:<40s}: {n_valid:3d} ({100*n_valid/len(df):5.1f}%)')

# Phase 1 complete-case count
phase1_complete = df[PHASE1_CORE].notna().all(axis=1)
print(f'\nPhase 1 complete-case N (all 5 core features): {phase1_complete.sum()}')
print(f'Condition breakdown within Phase 1 population:')
print(df.loc[phase1_complete, 'study_condition'].value_counts(dropna=False).to_string())


## C — Feature-Space Diagnostics


In [ ]:
# C02 — Univariate distributions of core and secondary features

diag_features = PHASE1_CORE + ['mu_hat', 'H_out', 'H_hour', 'H_app', 'chi_behav']
diag_features = [f for f in diag_features if f in df.columns]

n_cols = 4
n_rows = int(np.ceil(len(diag_features) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 3 * n_rows))
axes = axes.flatten()

for i, feat in enumerate(diag_features):
    vals = df[feat].dropna()
    axes[i].hist(vals, bins=30, alpha=0.7, color='steelblue', edgecolor='white')
    skew = vals.skew()
    axes[i].set_title(f'{feat}\n(n={len(vals)}, skew={skew:.2f})', fontsize=9)
    axes[i].set_xlabel('')
    axes[i].set_ylabel('Count')

for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

fig.suptitle('Univariate Distributions — Core and Secondary Features', fontsize=12, y=1.01)
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig0_univariate_distributions.png', bbox_inches='tight')
plt.close()
print('Saved fig0_univariate_distributions.png')

# Descriptive stats
df[diag_features].describe().round(3)


In [ ]:
# C03 — Pairwise correlation matrix (core + secondary features)

corr_features = PHASE1_CORE + ['mu_hat', 'H_SDE', 'beta_behav', 'H_out',
                                 'H_hour', 'H_app', 'chi_behav', 'H_trans', 'I_consec']
corr_features = [f for f in corr_features if f in df.columns]

corr_df = df[corr_features].dropna(subset=PHASE1_CORE).corr(method='pearson')

fig, ax = plt.subplots(figsize=(11, 9))
mask = np.triu(np.ones_like(corr_df, dtype=bool), k=1)
sns.heatmap(
    corr_df, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
    annot_kws={'size': 8}
)
ax.set_title('Feature Correlation Matrix (Phase 1 complete-case population)', fontsize=11)
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig1_feature_correlation_matrix.png', bbox_inches='tight')
plt.close()
print('Saved fig1_feature_correlation_matrix.png')

# Report key redundancy pairs
print('\nKey redundancy pairs (expected from blueprint):')
pairs = [('sig_hat','H_SDE'), ('sig_hat','beta_behav'), ('sig_hat','H_emp'),
         ('HHI_net','rho_self'),   # both retained as CORE — must be checked
         ('H_app','H_trans'), ('H_app','I_consec'), ('H_trans','I_consec'),
         ('x_mean','p_use_approx') if 'p_use_approx' in df.columns else None]
for p in pairs:
    if p and p[0] in corr_df.index and p[1] in corr_df.columns:
        print(f'  r({p[0]}, {p[1]}) = {corr_df.loc[p[0], p[1]]:.3f}')


In [ ]:
# C04 — PCA on all candidate features (diagnostic only — NOT used for clustering)

pca_features = [f for f in corr_features if f in df.columns]
pca_data = df[pca_features].dropna(subset=PHASE1_CORE)
pca_complete = pca_data[pca_features].dropna()

scaler_pca = StandardScaler()
X_pca_scaled = scaler_pca.fit_transform(pca_complete)

pca = PCA(n_components=min(len(pca_features), 8), random_state=RANDOM_STATE)
pca.fit(X_pca_scaled)

cumvar = np.cumsum(pca.explained_variance_ratio_)
n_80 = np.argmax(cumvar >= 0.80) + 1
print(f'PCA on {pca_complete.shape[0]} complete-case participants, {len(pca_features)} features')
print(f'Components to explain 80% variance: {n_80}')
print('Explained variance by component:')
for i, ev in enumerate(pca.explained_variance_ratio_):
    print(f'  PC{i+1}: {ev:.3f}  (cumulative: {cumvar[i]:.3f})')

# Biplot
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Scree
axes[0].bar(range(1, len(pca.explained_variance_ratio_) + 1),
            pca.explained_variance_ratio_, color='steelblue', alpha=0.8)
axes[0].plot(range(1, len(cumvar) + 1), cumvar, 'k-o', ms=4)
axes[0].axhline(0.80, color='red', ls='--', label='80% threshold')
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Explained Variance Ratio')
axes[0].set_title('PCA Scree Plot')
axes[0].legend(fontsize=8)

# Loadings on PC1 vs PC2
loadings = pd.DataFrame(
    pca.components_[:2].T,
    index=pca_features, columns=['PC1', 'PC2']
).sort_values('PC1')
axes[1].barh(range(len(loadings)), loadings['PC1'], alpha=0.7, label='PC1', color='steelblue')
axes[1].barh(range(len(loadings)), loadings['PC2'], alpha=0.5, label='PC2', color='coral')
axes[1].set_yticks(range(len(loadings)))
axes[1].set_yticklabels(loadings.index, fontsize=8)
axes[1].axvline(0, color='k', lw=0.5)
axes[1].set_xlabel('Loading')
axes[1].set_title('PC1 / PC2 Feature Loadings')
axes[1].legend(fontsize=8)

plt.tight_layout()
fig.savefig(FIG_DIR / 'fig2_pca_biplot.png', bbox_inches='tight')
plt.close()
print('Saved fig2_pca_biplot.png')


In [ ]:
# C05 — Missing data pattern for core features

miss_features = PHASE1_CORE + ['mu_hat', 'H_out', 'H_hour', 'H_app', 'chi_behav',
                                 'is_dys_eligible', 'four_model_convergent']
miss_features = [f for f in miss_features if f in df.columns]

miss_df = df[miss_features].isna().astype(int)
print('Missing data summary:')
for col in miss_features:
    n_miss = df[col].isna().sum()
    print(f'  {col:<45s}: {n_miss:3d} missing ({100*n_miss/len(df):.1f}%)')

# Test whether missingness correlates with condition
print('\nMissingness by condition (StoP features — sig_hat):')
miss_sig = df['sig_hat'].isna()
for cond in df['study_condition'].dropna().unique():
    mask_c = df['study_condition'] == cond
    n_cond = mask_c.sum()
    n_miss_c = (miss_sig & mask_c).sum()
    print(f'  {cond}: {n_miss_c}/{n_cond} missing ({100*n_miss_c/n_cond:.1f}%)')


## D — Feature Selection and Standardization


In [ ]:
# D01-D03 — Assemble Phase 1 matrix and standardize
#
# Phase 1 core (FLAG P1): sig_hat, H_emp, x_mean, HHI_net, rho_self
# Excluded per blueprint §3.2:
#   beta_behav, T_behav, H_SDE  -> monotone transforms of sig_hat
#   is_dys_eligible              -> binarization of sig_hat
#   four_model_convergent        -> composite of sig_hat + H_emp
#   p_use_approx                 -> near-duplicate of x_mean
#   I_consec                     -> derived (= H_app - H_trans)
#   sigma_mu_ratio               -> 31% NaN
#   top1_share_net, top3_share_net -> correlated with HHI_net
#   H_pi                         -> lower coverage (66.4%)
#   study_condition              -> overlay only, NOT a feature

# Assemble Phase 1 analysis population
# Include secondary features so Phase 2 code can access them
EXTRA_COLS = ['H_out', 'mu_hat', 'sigma_mu_ratio', 'is_dys_eligible',
               'behaviorally_deterministic', 'four_model_convergent', 'beta_behav',
               'chi_behav', 'F_behav', 'H_trans', 'I_consec', 'H_SDE',
               'regime_label', 'behavioral_regime', 'high_chi']
EXTRA_COLS = [c for c in EXTRA_COLS if c in df.columns]
ALL_P1_COLS = PHASE1_CORE + PHASE2_SENSITIVITY_EXTRA + EXTRA_COLS + ['participant_id_canonical', 'study_condition']
ALL_P1_COLS = list(dict.fromkeys(ALL_P1_COLS))  # deduplicate
ALL_P1_COLS = [c for c in ALL_P1_COLS if c in df.columns]
df_p1 = df[ALL_P1_COLS].dropna(subset=PHASE1_CORE).copy()
print(f'Phase 1 analysis population: N = {len(df_p1)}')
print(f'Condition distribution:')
print(df_p1['study_condition'].value_counts(dropna=False).to_string())

# Standardize (zero-mean, unit variance)
scaler = StandardScaler()
X_core = df_p1[PHASE1_CORE].values
X_scaled = scaler.fit_transform(X_core)

print(f'\nCore features after standardization:')
for i, feat in enumerate(PHASE1_CORE):
    print(f'  {feat}: mean={X_scaled[:,i].mean():.4f}, std={X_scaled[:,i].std():.4f}')


## E — Phase 1 Clustering (5 Core Features)


In [ ]:
# E01-E03 — K-means k=2..6 with evaluation metrics

K_RANGE = range(2, 7)
km_results = {}

for k in K_RANGE:
    km = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    ch  = calinski_harabasz_score(X_scaled, labels)
    db  = davies_bouldin_score(X_scaled, labels)
    cluster_counts = pd.Series(labels).value_counts().sort_index()
    min_size = cluster_counts.min()
    km_results[k] = {
        'model': km, 'labels': labels,
        'silhouette': sil, 'CH': ch, 'DB': db,
        'cluster_counts': cluster_counts, 'min_size': min_size
    }
    print(f'k={k}: silhouette={sil:.3f}, CH={ch:.1f}, DB={db:.3f}, '
          f'min_cluster_size={min_size}, counts={cluster_counts.values.tolist()}')


In [ ]:
# E04 — Hierarchical clustering and dendrogram

linkage_matrix = linkage(X_scaled, method='ward', metric='euclidean')

fig, ax = plt.subplots(figsize=(12, 5))
dend = dendrogram(
    linkage_matrix, ax=ax,
    truncate_mode='lastp', p=30,
    show_leaf_counts=True, leaf_rotation=45,
    color_threshold=None
)
ax.set_title('Hierarchical Clustering Dendrogram (Ward linkage, 5 core features)', fontsize=11)
ax.set_xlabel('Participant cluster')
ax.set_ylabel('Distance')
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig7_dendrogram.png', bbox_inches='tight')
plt.close()
print('Saved fig7_dendrogram.png')

# Hierarchical cluster assignments for k=2..5
hc_results = {}
for k in range(2, 6):
    hc_labels = fcluster(linkage_matrix, t=k, criterion='maxclust') - 1  # 0-indexed
    hc_sil = silhouette_score(X_scaled, hc_labels)
    hc_results[k] = {'labels': hc_labels, 'silhouette': hc_sil}
    print(f'HC k={k}: silhouette={hc_sil:.3f}, counts={pd.Series(hc_labels).value_counts().sort_index().values.tolist()}')


In [ ]:
# E05 — Gaussian Mixture Model k=2..5

gmm_results = {}
for k in range(2, 6):
    gmm = GaussianMixture(n_components=k, covariance_type='full',
                           n_init=5, random_state=RANDOM_STATE)
    gmm.fit(X_scaled)
    labels_gmm = gmm.predict(X_scaled)
    bic = gmm.bic(X_scaled)
    aic = gmm.aic(X_scaled)
    sil = silhouette_score(X_scaled, labels_gmm)
    gmm_results[k] = {'model': gmm, 'labels': labels_gmm, 'bic': bic, 'aic': aic, 'silhouette': sil}
    print(f'GMM k={k}: BIC={bic:.1f}, AIC={aic:.1f}, silhouette={sil:.3f}, '
          f'counts={pd.Series(labels_gmm).value_counts().sort_index().values.tolist()}')


In [ ]:
# E06-E08 — Plot cluster selection metrics and select k

ks = list(K_RANGE)
sil_scores = [km_results[k]['silhouette'] for k in ks]
ch_scores  = [km_results[k]['CH']         for k in ks]
db_scores  = [km_results[k]['DB']         for k in ks]
bic_scores = [gmm_results[k]['bic']       for k in range(2, 6)]

fig, axes = plt.subplots(1, 4, figsize=(14, 4))

axes[0].plot(ks, sil_scores, 'b-o'); axes[0].set_title('Silhouette (higher=better)')
axes[0].set_xlabel('k'); axes[0].set_ylabel('Score')

axes[1].plot(ks, ch_scores, 'g-o'); axes[1].set_title('Calinski-Harabasz (higher=better)')
axes[1].set_xlabel('k')

axes[2].plot(ks, db_scores, 'r-o'); axes[2].set_title('Davies-Bouldin (lower=better)')
axes[2].set_xlabel('k')

axes[3].plot(range(2, 6), bic_scores, 'k-o'); axes[3].set_title('GMM BIC (lower=better)')
axes[3].set_xlabel('k')

for ax in axes:
    ax.grid(alpha=0.3)

plt.tight_layout()
fig.savefig(FIG_DIR / 'fig3_cluster_selection_metrics.png', bbox_inches='tight')
plt.close()
print('Saved fig3_cluster_selection_metrics.png')

# ── Select k ─────────────────────────────────────────────
# Criteria (from blueprint §6.1):
#   1. Silhouette > 0.2
#   2. Dendrogram cut aligns with k-means solution
#   3. Min cluster size >= 40
#   4. Interpretable in terms of paper phenotype dimensions

print('\nCluster selection summary:')
print(f'{"k":>3} | {"silhouette":>10} | {"CH":>10} | {"DB":>8} | {"min_size":>9} | eligible')
print('-' * 60)
for k in ks:
    r = km_results[k]
    eligible = 'YES' if (r['silhouette'] > 0.2 and r['min_size'] >= 40) else 'no'
    print(f'{k:>3} | {r["silhouette"]:>10.3f} | {r["CH"]:>10.1f} | {r["DB"]:>8.3f} | {r["min_size"]:>9d} | {eligible}')

# Select k: prefer k with best silhouette among eligible solutions
eligible_ks = [k for k in ks if km_results[k]['silhouette'] > 0.2 and km_results[k]['min_size'] >= 40]
if eligible_ks:
    SELECTED_K = max(eligible_ks, key=lambda k: km_results[k]['silhouette'])
else:
    # Fallback: best silhouette regardless
    SELECTED_K = max(ks, key=lambda k: km_results[k]['silhouette'])
    print('WARNING: No solution meets all criteria; selecting by silhouette alone.')

print(f'\n>>> Selected k = {SELECTED_K} <<<')
print(f'    Silhouette: {km_results[SELECTED_K]["silhouette"]:.3f}')
print(f'    Cluster sizes: {km_results[SELECTED_K]["cluster_counts"].values.tolist()}')

# Assign final cluster labels (0-indexed, re-labeled alphabetically by x_mean)
FINAL_LABELS = km_results[SELECTED_K]['labels'].copy()

# Relabel clusters by ascending x_mean (Profile A = lowest override rate)
cluster_xmean = (
    pd.DataFrame({'cluster': FINAL_LABELS, 'x_mean': df_p1['x_mean'].values})
    .groupby('cluster')['x_mean'].mean()
    .sort_values()
)
relabel_map = {old: new for new, old in enumerate(cluster_xmean.index)}
FINAL_LABELS = np.array([relabel_map[l] for l in FINAL_LABELS])
profile_names = [f'Profile {chr(65+i)}' for i in range(SELECTED_K)]

df_p1['cluster_p1'] = FINAL_LABELS
df_p1['profile_p1'] = [profile_names[l] for l in FINAL_LABELS]

print(f'\nFinal cluster assignment (relabeled by ascending x_mean):')
print(df_p1['profile_p1'].value_counts().sort_index().to_string())


## F — Robustness Checks


In [ ]:
# F01 — Bootstrap stability (Jaccard coefficient)

def jaccard_stability(X, k, n_boot=100, sample_frac=0.80, seed=42):
    """Bootstrap Jaccard stability for k-means.
    Returns mean and std of per-bootstrap Jaccard coefficients.
    """
    rng = np.random.RandomState(seed)
    n = len(X)
    km_full = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=seed)
    labels_full = km_full.fit_predict(X)
    scores = []
    for b in range(n_boot):
        idx = rng.choice(n, int(n * sample_frac), replace=False)
        km_b = KMeans(n_clusters=k, init='k-means++', n_init=5, random_state=b)
        labels_b = km_b.fit_predict(X[idx])
        lf_sub = labels_full[idx]
        # Pairwise agreement (upper triangle only)
        triu = np.triu_indices(len(idx), k=1)
        same_full = lf_sub[triu[0]] == lf_sub[triu[1]]
        same_boot = labels_b[triu[0]] == labels_b[triu[1]]
        denom = np.sum(same_full | same_boot)
        j = np.sum(same_full & same_boot) / denom if denom > 0 else 0.0
        scores.append(j)
    return np.mean(scores), np.std(scores)

print(f'Bootstrap stability for selected k={SELECTED_K} (100 bootstraps, 80% sample):')
jac_mean, jac_std = jaccard_stability(X_scaled, SELECTED_K, n_boot=100, sample_frac=0.80)
print(f'  Jaccard mean = {jac_mean:.3f} (std = {jac_std:.3f})')
if jac_mean > 0.75:
    stability_verdict = 'STABLE (>0.75)'
elif jac_mean > 0.60:
    stability_verdict = 'MODERATE (0.60-0.75)'
else:
    stability_verdict = 'UNSTABLE (<0.60) — interpret with caution'
print(f'  Stability verdict: {stability_verdict}')

# Also report for neighboring k
for k_alt in [k for k in ks if k != SELECTED_K]:
    jm, js = jaccard_stability(X_scaled, k_alt, n_boot=50, sample_frac=0.80)
    print(f'  k={k_alt}: Jaccard={jm:.3f} (std={js:.3f})')


In [ ]:
# F02-F03 — Sensitivity to feature set: exclude H_emp, check agreement
#
# Rationale: H_emp and sig_hat have r=0.650 (partial redundancy).
# Test whether cluster structure is driven by this overlap.

SENS_FEATURES = [f for f in PHASE1_CORE if f != 'H_emp']  # 4-feature set
X_sens = StandardScaler().fit_transform(df_p1[SENS_FEATURES].values)

km_sens = KMeans(n_clusters=SELECTED_K, init='k-means++', n_init=10, random_state=RANDOM_STATE)
labels_sens = km_sens.fit_predict(X_sens)

ari_sens = adjusted_rand_score(FINAL_LABELS, labels_sens)
sil_sens = silhouette_score(X_sens, labels_sens)

print(f'Sensitivity test: clustering with {SENS_FEATURES} (dropping H_emp)')
print(f'  Silhouette (4-feature): {sil_sens:.3f}')
print(f'  ARI vs full 5-feature solution: {ari_sens:.3f}')
if ari_sens > 0.80:
    print('  -> High ARI: cluster structure is robust to removal of H_emp')
elif ari_sens > 0.60:
    print('  -> Moderate ARI: partial overlap; H_emp adds independent information')
else:
    print('  -> Low ARI: H_emp substantially changes cluster structure')

# Network-feature redundancy audit. HHI_net and rho_self are conceptually
# distinct, but both can approach one in single-app-dominated sequences.
r_net, p_net = stats.pearsonr(df_p1['HHI_net'], df_p1['rho_self'])
print(f'\nHHI_net vs rho_self: r={r_net:.3f}, p={p_net:.4g}, n={len(df_p1)}')
network_sensitivity = []
for dropped in ['HHI_net', 'rho_self']:
    features = [f for f in PHASE1_CORE if f != dropped]
    X_net_sens = StandardScaler().fit_transform(df_p1[features].values)
    labels = KMeans(n_clusters=SELECTED_K, init='k-means++', n_init=10,
                    random_state=RANDOM_STATE).fit_predict(X_net_sens)
    ari = adjusted_rand_score(FINAL_LABELS, labels)
    sil = silhouette_score(X_net_sens, labels)
    network_sensitivity.append(dict(dropped=dropped, ARI=ari, silhouette=sil))
    print(f'Drop {dropped}: silhouette={sil:.3f}; ARI vs full={ari:.3f}')
network_sensitivity_df = pd.DataFrame(network_sensitivity)
network_sensitivity_df.to_csv(OUTPUT_DIR / 'ds5_network_feature_sensitivity.csv', index=False)


## G — Cluster Profiles


In [ ]:
# G01-G05 — Per-cluster means on all features + interpretation overlays

# Merge cluster assignments back to full df for overlay features
df_full = df.merge(
    df_p1[['participant_id_canonical', 'cluster_p1', 'profile_p1']],
    on='participant_id_canonical', how='left'
)

# Features for profile table: core + secondary + interpretation-only
profile_features = (
    PHASE1_CORE
    + ['mu_hat', 'sigma_mu_ratio', 'H_out', 'H_hour', 'H_app',
       'chi_behav', 'F_behav', 'beta_behav', 'H_SDE',
       'H_trans', 'I_consec', 'top1_share_net',
       'deviation_from_predicted_fixed_point']
)
profile_features = [f for f in profile_features if f in df_full.columns]

p1_pop = df_full[df_full['cluster_p1'].notna()]

profile_stats = []
for feat in profile_features:
    row = {'feature': feat}
    for pname in profile_names:
        vals = p1_pop.loc[p1_pop['profile_p1'] == pname, feat].dropna()
        row[f'{pname}_mean']   = vals.mean()
        row[f'{pname}_median'] = vals.median()
        row[f'{pname}_std']    = vals.std()
        row[f'{pname}_n']      = len(vals)
    profile_stats.append(row)

profile_df = pd.DataFrame(profile_stats)

# Print key core features
print('Per-cluster means on core features:')
print(f'{"Feature":<40}', end='')
for pname in profile_names:
    print(f'{pname:>14}', end='')
print()
print('-' * (40 + 14 * len(profile_names)))
for feat in PHASE1_CORE:
    row = profile_df[profile_df['feature'] == feat].iloc[0]
    print(f'{feat:<40}', end='')
    for pname in profile_names:
        print(f'{row[f"{pname}_mean"]:>14.3f}', end='')
    print()

# Interpretation-only overlay
print('\nInterpretation-only overlays (proportions):')
for flag_col in ['is_dys_eligible', 'behaviorally_deterministic', 'four_model_convergent', 'high_chi']:
    if flag_col not in df_full.columns:
        continue
    print(f'  {flag_col}:')
    for pname in profile_names:
        sub = p1_pop.loc[p1_pop['profile_p1'] == pname, flag_col].dropna()
        if len(sub) > 0:
            pct = 100 * sub.astype(float).mean()
            print(f'    {pname}: {pct:.1f}% ({len(sub)} with data)')


In [ ]:
# G06 — Radar chart: cluster profiles across paper phenotype dimensions

# Map paper dimensions to normalized cluster means
RADAR_DIMS = {
    'Override\ntendency': 'x_mean',
    'Impulsivity\n(noise)': 'sig_hat',
    'Behavioral\nentropy': 'H_emp',
    'Network\nconcentration': 'HHI_net',
    'Habitual\nrepetition': 'rho_self',
    'Temporal\nspread': 'H_hour',
    'Susceptibility': 'chi_behav',
}
RADAR_DIMS = {k: v for k, v in RADAR_DIMS.items() if v in df_full.columns}

dim_labels = list(RADAR_DIMS.keys())
dim_feats  = list(RADAR_DIMS.values())
N_dim = len(dim_labels)

# Normalize each dimension to [0,1] across the Phase 1 population
norm_vals = {}
for feat in dim_feats:
    vals = p1_pop[feat]
    vmin, vmax = vals.min(), vals.max()
    norm_vals[feat] = (vals - vmin) / (vmax - vmin + 1e-12)

angles = np.linspace(0, 2 * np.pi, N_dim, endpoint=False).tolist()
angles += angles[:1]  # close the polygon

colors = plt.cm.Set1(np.linspace(0, 0.8, len(profile_names)))

fig, ax = plt.subplots(1, 1, figsize=(7, 7), subplot_kw=dict(polar=True))

for i, pname in enumerate(profile_names):
    sub = p1_pop[p1_pop['profile_p1'] == pname]
    vals_norm = [norm_vals[f][sub.index].mean() for f in dim_feats]
    vals_norm += vals_norm[:1]
    ax.plot(angles, vals_norm, 'o-', color=colors[i], lw=2, label=pname)
    ax.fill(angles, vals_norm, color=colors[i], alpha=0.10)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(dim_labels, fontsize=9)
ax.set_yticklabels([])
ax.set_title('Cluster Profiles across Phenotype Dimensions\n(normalized to [0,1])', fontsize=11, pad=15)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=9)

plt.tight_layout()
plt.show()
fig.savefig(FIG_DIR / 'fig4_cluster_profiles_radar.png', bbox_inches='tight')
#plt.close()
print('Saved fig4_cluster_profiles_radar.png')


In [ ]:
# G06b — Enhanced radar chart: boxed axis labels, section layer bands, thin model arcs

from collections import OrderedDict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


# ─────────────────────────────────────────────
# 0.  DIMENSION & MODEL DEFINITIONS
# ─────────────────────────────────────────────

RADAR_DIMS = OrderedDict([
    ('Behavioral\nentropy',    'H_emp'),      # InT
    ('Impulsivity\n(noise)',   'sig_hat'),    # StoP — behavioral volatility
    ('Override\ntendency',     'x_mean'),     # raw override ratio
    ('Susceptibility',         'chi_behav'),  # StM
    ('Temporal\nspread',       'H_hour'),     # InT
    ('Habitual\nrepetition',   'rho_self'),   # NeT
    ('Network\nconcentration', 'HHI_net'),    # NeT
])

AXIS_MODEL = {
    'Behavioral\nentropy':    'InT',
    'Impulsivity\n(noise)':   'StoP',
    'Override\ntendency':     'StoP',
    'Susceptibility':         'StM',
    'Temporal\nspread':       'InT',
    'Habitual\nrepetition':   'NeT',
    'Network\nconcentration': 'NeT',
}

MODEL_COLORS = {
    'InT':  '#2196F3',
    'StoP': '#FF9800',
    'StM':  '#9C27B0',
    'NeT':  '#4CAF50',
}

PROFILE_COLORS = {'Profile A': '#A20A7C', 'Profile B': '#7E57C2'}

# Three conceptual layers — indices into RADAR_DIMS order
SECTION_LAYERS = [
    {'name': 'Behavioral dynamics',        'dim_idx': [0,1,2], 'color': '#C62828'},
    {'name': 'Theoretical interpretation', 'dim_idx': [3,4],   'color': "#15B5C0"},
    {'name': 'Network structure',          'dim_idx': [5,6],   'color': "#CC1B97"},
]


# ─────────────────────────────────────────────
# 1.  NORMALIZATION
# ─────────────────────────────────────────────

def normalize_features(df, dim_feats):
    norm = {}
    for feat in dim_feats:
        v = df[feat].dropna()
        norm[feat] = (df[feat] - v.min()) / (v.max() - v.min() + 1e-12)
    return norm


# ─────────────────────────────────────────────
# 2.  THIN MODEL-FAMILY ARCS  (no labels, reduced intensity)
# ─────────────────────────────────────────────

def draw_model_arcs(ax, angles, dim_labels, axis_model, model_colors,
                    arc_r=1.04, arc_lw=3.5, alpha=0.45):
    """Thin colored arc per axis — color family hint only, no text labels."""
    n = len(angles)
    half_gap = (2 * np.pi / n) * 0.40
    for label, angle in zip(dim_labels, angles):
        model = axis_model.get(label, '')
        color = model_colors.get(model, '#AAAAAA')
        theta = np.linspace(angle - half_gap, angle + half_gap, 40)
        ax.plot(theta, np.full_like(theta, arc_r),
                color=color, lw=arc_lw, solid_capstyle='round',
                alpha=alpha, clip_on=False, zorder=10)


# ─────────────────────────────────────────────
# 3.  SECTION LAYER BANDS
# ─────────────────────────────────────────────

def draw_section_layers(ax, angles, section_layers,
                        band_inner=1.09, band_outer=1.20, label_r=1.145):
    """
    Colored arc shadow band per conceptual layer with a centered label.
    Three layers:
      - Behavioral dynamics        (Behavioral entropy, Impulsivity, Override tendency)
      - Theoretical interpretation (Susceptibility, Temporal spread)
      - Network structure          (Habitual repetition, Network concentration)
    """
    step = angles[1] - angles[0]
    half = step / 2

    for sec in section_layers:
        idx   = sec['dim_idx']
        color = sec['color']
        a0    = angles[idx[0]]  - half
        a1    = angles[idx[-1]] + half
        theta = np.linspace(a0, a1, 300)

        ax.fill_between(theta, band_inner, band_outer,
                        color=color, alpha=0.13, clip_on=False, zorder=5)
        ax.plot(theta, np.full_like(theta, band_inner),
                color=color, lw=0.8, alpha=0.40, clip_on=False, zorder=6)
        ax.plot(theta, np.full_like(theta, band_outer),
                color=color, lw=0.8, alpha=0.40, clip_on=False, zorder=6)

        mid = (a0 + a1) / 2
        ax.text(mid, label_r, sec['name'],
                ha='center', va='center',
                fontsize=7.5, fontweight='bold', color=color,
                clip_on=False, zorder=21, multialignment='center')


# ─────────────────────────────────────────────
# 4.  BOXED AXIS LABELS  (above section bands)
# ─────────────────────────────────────────────

def draw_axis_label_boxes(ax, angles, dim_labels, label_r=1.33, fontsize=9.5):
    """Place each axis label in a white rounded box outside the section bands."""
    for label, angle in zip(dim_labels, angles):
        ax.text(angle, label_r, label,
                ha='center', va='center',
                fontsize=fontsize, color='#222222',
                clip_on=False, zorder=25, multialignment='center',
                bbox=dict(boxstyle='round,pad=0.30', fc='white',
                          ec='#BBBBBB', lw=0.9, alpha=0.97))


# ─────────────────────────────────────────────
# 5.  IQR BAND HELPER
# ─────────────────────────────────────────────

def compute_iqr_band(sub_df, norm_cols, angles):
    q25  = sub_df[norm_cols].quantile(0.25).to_numpy(dtype=float)
    q75  = sub_df[norm_cols].quantile(0.75).to_numpy(dtype=float)
    mean = sub_df[norm_cols].mean(skipna=True).to_numpy(dtype=float)
    def close(arr): return np.append(arr, arr[0])
    return close(q25), close(q75), close(mean)


# ─────────────────────────────────────────────
# 6.  SINGLE PROFILE
# ─────────────────────────────────────────────

def plot_single_profile(ax, sub_df, norm_cols, angles, profile_name,
                        color, dim_labels, axis_model, model_colors,
                        section_layers, individual_alpha=0.10, individual_lw=0.6,
                        mean_lw=2.8, show_annotations=False):

    for _, row in sub_df.iterrows():
        v = np.append(row[norm_cols].to_numpy(dtype=float), row[norm_cols].iloc[0])
        ax.plot(angles, v, color=color, alpha=individual_alpha, lw=individual_lw, zorder=1)

    q25, q75, mean_vals = compute_iqr_band(sub_df, norm_cols, angles)
    ax.fill_between(angles, q25, q75, color=color, alpha=0.22, zorder=2)
    ax.plot(angles, q25, color=color, alpha=0.45, lw=1.0, ls='--', zorder=4)
    ax.plot(angles, q75, color=color, alpha=0.45, lw=1.0, ls='--', zorder=4)

    ax.plot(angles, mean_vals, 'o-', color=color, lw=mean_lw, ms=5,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=1.8,
            label=f'{profile_name} mean  (n={len(sub_df)})', zorder=5)
    ax.fill(angles, mean_vals, color=color, alpha=0.10, zorder=3)

    draw_model_arcs(ax, angles[:-1], dim_labels, axis_model, model_colors)
    draw_section_layers(ax, angles[:-1], section_layers)
    draw_axis_label_boxes(ax, angles[:-1], dim_labels)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([''] * len(dim_labels))
    ax.tick_params(axis='x', length=0)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.00])
    ax.set_yticklabels(['0.25','0.50','0.75','1.00'],
                       fontsize=7.5, alpha=0.5, color='#555555')
    ax.spines['polar'].set_color('#CCCCCC')
    ax.grid(color='#DDDDDD', lw=0.6)

    if show_annotations:
        net_idx   = [i for i,l in enumerate(dim_labels) if axis_model.get(l,'')=='NeT']
        behav_idx = [i for i,l in enumerate(dim_labels) if axis_model.get(l,'') in ('StoP','InT')]
        ax.annotate('structural\nspike',
                    xy=(angles[net_idx[0]], q75[net_idx[0]]),
                    xytext=(angles[net_idx[0]] + 0.35, 0.82),
                    fontsize=7.5, color='#2E7D32', fontweight='bold',
                    ha='center', annotation_clip=False,
                    arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.1))
        ax.annotate('behavioral\naxes flat',
                    xy=(angles[behav_idx[1]], q75[behav_idx[1]]),
                    xytext=(angles[behav_idx[1]] - 0.45, 0.62),
                    fontsize=7.5, color='#C62828', fontweight='bold',
                    ha='center', annotation_clip=False,
                    arrowprops=dict(arrowstyle='->', color='#C62828', lw=1.1))


# ─────────────────────────────────────────────
# 7.  MAIN FIGURE
# ─────────────────────────────────────────────

def plot_phenotype_profiles_enhanced(p1_pop, profile_col='profile_p1',
                                     participant_col='participant_id_canonical',
                                     save_path=None, figsize=(18, 9.5)):

    available  = OrderedDict((k,v) for k,v in RADAR_DIMS.items() if v in p1_pop.columns)
    dim_labels = list(available.keys())
    dim_feats  = list(available.values())
    am         = {k: AXIS_MODEL[k] for k in dim_labels}

    norm_map = normalize_features(p1_pop, dim_feats)
    p1_radar = p1_pop.copy()
    for feat in dim_feats:
        p1_radar[f'{feat}_norm'] = norm_map[feat]
    norm_cols = [f'{feat}_norm' for feat in dim_feats]

    angles = np.linspace(0, 2*np.pi, len(dim_labels), endpoint=False)
    angles = np.append(angles, angles[0])

    fig, axes = plt.subplots(1, 2, figsize=figsize, subplot_kw=dict(polar=True))
    fig.subplots_adjust(wspace=0.7, left=0.08, right=0.92, top=0.85, bottom=0.12)

    specs = [('Profile A', PROFILE_COLORS['Profile A'], axes[0], False),
             ('Profile B', PROFILE_COLORS['Profile B'], axes[1], True)]

    for pname, color, ax, annotate in specs:
        sub = p1_radar[p1_radar[profile_col] == pname].copy()
        if sub.empty:
            ax.set_title(f'{pname}\n(no data)', fontsize=11)
            continue
        plot_single_profile(ax=ax, sub_df=sub, norm_cols=norm_cols, angles=angles,
                            profile_name=pname, color=color, dim_labels=dim_labels,
                            axis_model=am, model_colors=MODEL_COLORS,
                            section_layers=SECTION_LAYERS, show_annotations=annotate)
        n = sub[participant_col].nunique() if participant_col in sub.columns else len(sub)
        lbl = '(a)' if 'A' in pname else '(b)'
        ax.set_title(f'{lbl}  {pname}\nn = {n} participants',
                     fontsize=11, fontweight='bold', pad=22, color='#222222')
        ax.legend(loc='upper right', bbox_to_anchor=(1.38, 1.15),
                  fontsize=9, framealpha=0.9)

    def _desc(m):
        return {'InT':'Information Theory','StoP':'Stochastic Processes',
                'StM':'Statistical Mechanics','NeT':'Network Theory'}.get(m,m)

    arc_patches = [mpatches.Patch(color=c, label=f'{m}  —  {_desc(m)}')
                   for m,c in MODEL_COLORS.items()]
    arc_patches.append(mpatches.Patch(facecolor='grey', alpha=0.35,
                                      label='IQR band (25th–75th pct.)'))

    sec_patches = [mpatches.Patch(color=s['color'], alpha=0.55,
                                  label=s['name'].replace('\n',' '))
                   for s in SECTION_LAYERS]

    leg1 = fig.legend(handles=arc_patches, loc='lower left',
                      bbox_to_anchor=(0.03, -0.03), ncol=1,
                      fontsize=8.5, framealpha=0.95, edgecolor='#CCCCCC',
                      title='Model families  (axis color)', title_fontsize=8.5)
    fig.add_artist(leg1)
    fig.legend(handles=sec_patches, loc='lower right',
               bbox_to_anchor=(0.97, -0.03), ncol=1,
               fontsize=8.5, framealpha=0.95, edgecolor='#CCCCCC',
               title='Conceptual layers  (band color)', title_fontsize=8.5)

    fig.suptitle('Multidimensional Behavioral Phenotype Profiles\n'
                 'Cluster means, IQR bands, and individual participant trajectories',
                 fontsize=13, fontweight='bold', y=0.97, color='#111111')

    if save_path is not None:
        fig.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f'Saved: {save_path}')
    plt.show()
    return fig, axes


# ─────────────────────────────────────────────
# 8.  CALL
# ─────────────────────────────────────────────

fig, axes = plot_phenotype_profiles_enhanced(
    p1_pop=p1_pop,
    profile_col='profile_p1',
    participant_col='participant_id_canonical',
    save_path=FIG_DIR / 'fig4_finalphenotype_profiles_enhanced.png',
)


In [ ]:
# ─────────────────────────────────────────────
# FONT SIZE CONFIGURATION (Adjust these to find what works best!)
# ─────────────────────────────────────────────
FONT_SIZES = {
    'figure_title':     20.0,  # Main title at the very top
    'subplot_title':    1.0,  # "Profile A / Profile B" headers
    'axis_labels':      18,  # Text inside the white rounded boxes
    'section_layers':   18,  # Text inside the colored layer bands
    'grid_ticks':        15,  # The 0.25, 0.50, 0.75, 1.00 radial marks
    'annotations':       15,  # Text for arrows ("structural spike", etc.)
    'subplot_legend':    18,  # The profile mean legends on the right of subplots
    'bottom_legends':    18,  # The big model family/conceptual layer blocks
    'legend_titles':     18,  # Heading text inside those bottom legend blocks
}

# ─────────────────────────────────────────────
# G06b — Enhanced radar chart: boxed axis labels, section layer bands, thin model arcs
# ─────────────────────────────────────────────

from collections import OrderedDict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


# ─────────────────────────────────────────────
# 0.  DIMENSION & MODEL DEFINITIONS
# ─────────────────────────────────────────────

RADAR_DIMS = OrderedDict([
    ('Behavioral\nentropy',    'H_emp'),      # InT
    ('Impulsivity\n(noise)',   'sig_hat'),    # StoP — behavioral volatility
    ('Override\ntendency',     'x_mean'),     # raw override ratio
    ('Susceptibility',         'chi_behav'),  # StM
    ('Temporal\nspread',       'H_hour'),     # InT
    ('Habitual\nrepetition',   'rho_self'),   # NeT
    ('Network\nconcentration', 'HHI_net'),    # NeT
])

AXIS_MODEL = {
    'Behavioral\nentropy':    'InT',
    'Impulsivity\n(noise)':   'StoP',
    'Override\ntendency':     'StoP',
    'Susceptibility':         'StM',
    'Temporal\nspread':       'InT',
    'Habitual\nrepetition':   'NeT',
    'Network\nconcentration': 'NeT',
}

MODEL_COLORS = {
    'InT':  '#2196F3',
    'StoP': '#FF9800',
    'StM':  "#C5D307",
    'NeT':  "#2AF431",
}

PROFILE_COLORS = {'Profile A': '#A20A7C', 'Profile B': '#7E57C2'}

# Three conceptual layers — indices into RADAR_DIMS order
SECTION_LAYERS = [
    {'name': 'Behavioral dynamics',        'dim_idx': [0,1,2], 'color': '#C62828'},
    {'name': 'Theoretical interpretation', 'dim_idx': [3,4],   'color': "#0A4B96"},
    {'name': 'Network structure',          'dim_idx': [5,6],   'color': "#044B07"},
]


# ─────────────────────────────────────────────
# 1.  NORMALIZATION
# ─────────────────────────────────────────────

def normalize_features(df, dim_feats):
    norm = {}
    for feat in dim_feats:
        v = df[feat].dropna()
        norm[feat] = (df[feat] - v.min()) / (v.max() - v.min() + 1e-12)
    return norm


# ─────────────────────────────────────────────
# 2.  THIN MODEL-FAMILY ARCS  (no labels, reduced intensity)
# ─────────────────────────────────────────────

def draw_model_arcs(ax, angles, dim_labels, axis_model, model_colors,
                    arc_r=1.04, arc_lw=3.5, alpha=0.45):
    """Thin colored arc per axis — color family hint only, no text labels."""
    n = len(angles)
    half_gap = (2 * np.pi / n) * 0.40
    for label, angle in zip(dim_labels, angles):
        model = axis_model.get(label, '')
        color = model_colors.get(model, '#AAAAAA')
        theta = np.linspace(angle - half_gap, angle + half_gap, 40)
        ax.plot(theta, np.full_like(theta, arc_r),
                color=color, lw=arc_lw, solid_capstyle='round',
                alpha=alpha, clip_on=False, zorder=10)


# ─────────────────────────────────────────────
# 3.  SECTION LAYER BANDS
# ─────────────────────────────────────────────

def draw_section_layers(ax, angles, section_layers,
                        band_inner=1.09, band_outer=1.20, label_r=1.145):
    """
    Colored arc shadow band per conceptual layer with a centered label.
    Three layers:
      - Behavioral dynamics        (Behavioral entropy, Impulsivity, Override tendency)
      - Theoretical interpretation (Susceptibility, Temporal spread)
      - Network structure          (Habitual repetition, Network concentration)
    """
    step = angles[1] - angles[0]
    half = step / 2

    for sec in section_layers:
        idx   = sec['dim_idx']
        color = sec['color']
        a0    = angles[idx[0]]  - half
        a1    = angles[idx[-1]] + half
        theta = np.linspace(a0, a1, 300)

        ax.fill_between(theta, band_inner, band_outer,
                        color=color, alpha=0.13, clip_on=False, zorder=5)
        ax.plot(theta, np.full_like(theta, band_inner),
                color=color, lw=0.8, alpha=0.40, clip_on=False, zorder=6)
        ax.plot(theta, np.full_like(theta, band_outer),
                color=color, lw=0.8, alpha=0.40, clip_on=False, zorder=6)

        mid = (a0 + a1) / 2
        #ax.text(mid, label_r, sec['name'],
        #        ha='center', va='center',
        #        fontsize=FONT_SIZES['section_layers'], fontweight='bold', color=color,
        #        clip_on=False, zorder=21, multialignment='center')


# ─────────────────────────────────────────────
# 4.  BOXED AXIS LABELS  (above section bands)
# ─────────────────────────────────────────────

def draw_axis_label_boxes(ax, angles, dim_labels, label_r=1.33):
    """Place each axis label in a white rounded box outside the section bands."""
    for label, angle in zip(dim_labels, angles):
        ax.text(angle, label_r, label,
                ha='center', va='center',
                fontsize=FONT_SIZES['axis_labels'], color='#222222',
                clip_on=False, zorder=25, multialignment='center',
                bbox=dict(boxstyle='round,pad=0.30', fc='white',
                          ec='#BBBBBB', lw=0.9, alpha=0.97))


# ─────────────────────────────────────────────
# 5.  IQR BAND HELPER
# ─────────────────────────────────────────────

def compute_iqr_band(sub_df, norm_cols, angles):
    q25  = sub_df[norm_cols].quantile(0.25).to_numpy(dtype=float)
    q75  = sub_df[norm_cols].quantile(0.75).to_numpy(dtype=float)
    mean = sub_df[norm_cols].mean(skipna=True).to_numpy(dtype=float)
    def close(arr): return np.append(arr, arr[0])
    return close(q25), close(q75), close(mean)


# ─────────────────────────────────────────────
# 6.  SINGLE PROFILE
# ─────────────────────────────────────────────

def plot_single_profile(ax, sub_df, norm_cols, angles, profile_name,
                        color, dim_labels, axis_model, model_colors,
                        section_layers, individual_alpha=0.20, individual_lw=0.6,
                        mean_lw=2.8, show_annotations=False):

    for _, row in sub_df.iterrows():
        v = np.append(row[norm_cols].to_numpy(dtype=float), row[norm_cols].iloc[0])
        ax.plot(angles, v, color=color, alpha=individual_alpha, lw=individual_lw, zorder=1)

    q25, q75, mean_vals = compute_iqr_band(sub_df, norm_cols, angles)
    ax.fill_between(angles, q25, q75, color=color, alpha=0.22, zorder=2)
    ax.plot(angles, q25, color=color, alpha=0.45, lw=1.0, ls='--', zorder=4)
    ax.plot(angles, q75, color=color, alpha=0.45, lw=1.0, ls='--', zorder=4)

    ax.plot(angles, mean_vals, 'o-', color=color, lw=mean_lw, ms=5,
            markerfacecolor='white', markeredgecolor=color, markeredgewidth=1.8,
            label=f'{profile_name} mean  (n={len(sub_df)})', zorder=5)
    ax.fill(angles, mean_vals, color=color, alpha=0.10, zorder=3)

    draw_model_arcs(ax, angles[:-1], dim_labels, axis_model, model_colors)
    draw_section_layers(ax, angles[:-1], section_layers)
    draw_axis_label_boxes(ax, angles[:-1], dim_labels)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels([''] * len(dim_labels))
    ax.tick_params(axis='x', length=0)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.00])
    ax.set_yticklabels(['0.25','0.50','0.75','1.00'],
                       fontsize=FONT_SIZES['grid_ticks'], alpha=0.5, color='#555555')
    #ax.spines['polar'].set_color('#CCCCCC')
    ax.spines['polar'].set_color("#272727")
    #ax.grid(color='#DDDDDD', lw=0.6)
    ax.grid(color="#121111", lw=0.6)

    var1 = 1
    if var1 == 0:
    #if show_annotations:
        net_idx   = [i for i,l in enumerate(dim_labels) if axis_model.get(l,'')=='NeT']
        behav_idx = [i for i,l in enumerate(dim_labels) if axis_model.get(l,'') in ('StoP','InT')]
        ax.annotate('structural\nspike',
                    xy=(angles[net_idx[0]], q75[net_idx[0]]),
                    xytext=(angles[net_idx[0]] + 0.35, 0.82),
                    fontsize=FONT_SIZES['annotations'], color='#2E7D32', fontweight='bold',
                    ha='center', annotation_clip=False,
                    arrowprops=dict(arrowstyle='->', color='#2E7D32', lw=1.1))
        ax.annotate('behavioral\naxes flat',
                    xy=(angles[behav_idx[1]], q75[behav_idx[1]]),
                    xytext=(angles[behav_idx[1]] - 0.45, 0.62),
                    fontsize=FONT_SIZES['annotations'], color='#C62828', fontweight='bold',
                    ha='center', annotation_clip=False,
                    arrowprops=dict(arrowstyle='->', color='#C62828', lw=1.1))


# ─────────────────────────────────────────────
# 7.  MAIN FIGURE
# ─────────────────────────────────────────────

def plot_phenotype_profiles_enhanced(p1_pop, profile_col='profile_p1',
                                     participant_col='participant_id_canonical',
                                     save_path=None, figsize=(18, 9.5)):

    available  = OrderedDict((k,v) for k,v in RADAR_DIMS.items() if v in p1_pop.columns)
    dim_labels = list(available.keys())
    dim_feats  = list(available.values())
    am         = {k: AXIS_MODEL[k] for k in dim_labels}

    norm_map = normalize_features(p1_pop, dim_feats)
    p1_radar = p1_pop.copy()
    for feat in dim_feats:
        p1_radar[f'{feat}_norm'] = norm_map[feat]
    norm_cols = [f'{feat}_norm' for feat in dim_feats]

    angles = np.linspace(0, 2*np.pi, len(dim_labels), endpoint=False)
    angles = np.append(angles, angles[0])

    fig, axes = plt.subplots(1, 2, figsize=figsize, subplot_kw=dict(polar=True))
    fig.subplots_adjust(wspace=0.7, left=0.08, right=0.92, top=0.85, bottom=0.12)

    specs = [('Profile A', PROFILE_COLORS['Profile A'], axes[0], False),
             ('Profile B', PROFILE_COLORS['Profile B'], axes[1], True)]

    for pname, color, ax, annotate in specs:
        sub = p1_radar[p1_radar[profile_col] == pname].copy()
        if sub.empty:
            ax.set_title(f'{pname}\n(no data)', fontsize=FONT_SIZES['subplot_title'])
            continue
        plot_single_profile(ax=ax, sub_df=sub, norm_cols=norm_cols, angles=angles,
                            profile_name=pname, color=color, dim_labels=dim_labels,
                            axis_model=am, model_colors=MODEL_COLORS,
                            section_layers=SECTION_LAYERS, show_annotations=annotate)
        n = sub[participant_col].nunique() if participant_col in sub.columns else len(sub)
        lbl = '(a)' if 'A' in pname else '(b)'
        #ax.set_title(f'{lbl}  {pname}\nn = {n} participants',fontsize=FONT_SIZES['subplot_title'], fontweight='bold', pad=22, color='#222222')
        #ax.legend(loc='upper right', bbox_to_anchor=(1.38, 1.15), fontsize=FONT_SIZES['subplot_legend'], framealpha=0.9)

    def _desc(m):
        return {'InT':'Information Theory','StoP':'Stochastic Processes',
                'StM':'Statistical Mechanics','NeT':'Network Theory'}.get(m,m)

    graph_patches = []  # ← initialize the list

    for pname, color, ax, annotate in specs:
        sub = p1_radar[p1_radar[profile_col] == pname].copy()
        if sub.empty:
            ax.set_title(f'{pname}\n(no data)', fontsize=FONT_SIZES['subplot_title'])
            continue
        n = sub[participant_col].nunique() if participant_col in sub.columns else len(sub)
        lbl = '(a)' if 'A' in pname else '(b)'

        # ← add a mean patch per profile, inside the loop
        graph_patches.append(mpatches.Patch(
            facecolor=color,
            label=f'{pname} mean'
            #label=f'{pname} mean  (n = {n} participants)'
        ))

    graph_patches.append(mpatches.Patch(facecolor='grey', alpha=0.35,
                                      label='IQR band (25th–75th pct.)'))
    
    arc_patches = [mpatches.Patch(color=c, label=f'{m}  —  {_desc(m)}')
                   for m,c in MODEL_COLORS.items()]
    
    sec_patches = [mpatches.Patch(color=s['color'], alpha=0.55,
                                  label=s['name'].replace('\n',' '))
                   for s in SECTION_LAYERS]
    
    #leg0 = fig.legend(handles=graph_patches,loc='lower left', bbox_to_anchor=(0.03, -0.03), ncol=1, 
    leg0 = fig.legend(handles=graph_patches,loc='upper center', bbox_to_anchor=(0.53, 0.9), ncol=1, 
                  fontsize=FONT_SIZES['bottom_legends'], framealpha=0.95, edgecolor='#CCCCCC',
                  title='Profiles  (mean line + IQR)', title_fontsize=FONT_SIZES['legend_titles'])
    fig.add_artist(leg0)

    leg1 = fig.legend(handles=arc_patches, loc='lower center',
                      bbox_to_anchor=(0.55, -0.04), ncol=1,
                      fontsize=FONT_SIZES['bottom_legends'], framealpha=0.95, edgecolor='#CCCCCC',
                      title='Model families  (axis color)', title_fontsize=FONT_SIZES['legend_titles'])
    fig.add_artist(leg1)
    
    fig.legend(handles=sec_patches, loc='lower right',
               bbox_to_anchor=(1, -0.04), ncol=1,
               fontsize=FONT_SIZES['bottom_legends'], framealpha=0.95, edgecolor='#CCCCCC',
               title='Conceptual layers  (band color)', title_fontsize=FONT_SIZES['legend_titles'])

    #fig.suptitle('Multidimensional Behavioral Phenotype Profiles\n'
    #             'Cluster means and individual participant trajectories',
    #             fontsize=FONT_SIZES['figure_title'], fontweight='bold', y=0.97, color='#111111')

    if save_path is not None:
        fig.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f'Saved: {save_path}')
    plt.show()
    return fig, axes


# ─────────────────────────────────────────────
# 8.  CALL
# ─────────────────────────────────────────────

fig, axes = plot_phenotype_profiles_enhanced(
    p1_pop=p1_pop,
    profile_col='profile_p1',
    participant_col='participant_id_canonical',
    save_path=FIG_DIR / 'fig4_fontphenotype_profiles_enhanced_notitle.png',
)


In [ ]:
# G06b.2.1 — Enhanced radar chart: profiles with IQR bands, model-family arcs, annotations

from collections import OrderedDict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches


# ─────────────────────────────────────────────
# 0.  RADAR DIMENSIONS
#     Three conceptual layers, grouped by model family for contiguous arcs:
#       Network structure     → NeT  (Network concentration, Habitual repetition)
#       Behavioral dynamics   → InT  (Behavioral entropy, Temporal spread)
#                             → StoP (Impulsivity/noise, Override tendency)
#       Theoretical interp.   → StM  (Susceptibility)
# ─────────────────────────────────────────────

RADAR_DIMS = OrderedDict([
    ('Behavioral\nentropy',    'H_emp'),      # InT  — H of daily override-rate dist.
    ('Impulsivity\n(noise)',   'sig_hat'),    # StoP — diffusion σ̂ (behavioral volatility)
    ('Override\ntendency',     'x_mean'),     # raw  — mean daily override rate (x̄)
    ('Susceptibility',         'chi_behav'),  # StM  — χ=β·p·(1−p); peaks near threshold
    ('Temporal\nspread',       'H_hour'),     # InT  — H of events across hours
    ('Habitual\nrepetition',   'rho_self'),   # NeT  — self-loop rate (same-app follow)
    ('Network\nconcentration', 'HHI_net'),    # NeT  — HHI of app-transition flows
])

AXIS_MODEL = {
    'Behavioral\nentropy':    'InT',
    'Impulsivity\n(noise)':   'StoP',
    'Override\ntendency':     'StoP',
    'Susceptibility':         'StM',
    'Temporal\nspread':       'InT',
    'Habitual\nrepetition':   'NeT',
    'Network\nconcentration': 'NeT',
}

MODEL_COLORS = {
    'InT':  '#2196F3',   # blue
    'StoP': '#FF9800',   # orange
    'StM':  '#9C27B0',   # purple
    'NeT':  '#4CAF50',   # green
}

PROFILE_COLORS = {'Profile A': '#A20A7C', 'Profile B': '#7E57C2'}


# ─────────────────────────────────────────────
# 1.  NORMALISATION
# ─────────────────────────────────────────────

def normalize_features(df, dim_feats):
    """Min-max normalize each feature across the full Phase 1 population."""
    norm = {}
    for feat in dim_feats:
        v = df[feat].dropna()
        norm[feat] = (df[feat] - v.min()) / (v.max() - v.min() + 1e-12)
    return norm


# ─────────────────────────────────────────────
# 2.  ARC DRAWING  (polar coordinates)
# ─────────────────────────────────────────────

def draw_model_arcs(ax, angles, dim_labels, axis_model, model_colors,
                    arc_r=1.18, arc_lw=6, label_r=1.34, fontsize=9):
    """
    Colored arc segment per axis (polar coords), grouped label per model family.
    """
    n = len(angles)
    half_gap = (2 * np.pi / n) * 0.38
    family_angles = {}

    for label, angle in zip(dim_labels, angles):
        model = axis_model.get(label, '')
        color = model_colors.get(model, '#AAAAAA')

        theta = np.linspace(angle - half_gap, angle + half_gap, 40)
        ax.plot(theta, np.full_like(theta, arc_r),
                color=color, lw=arc_lw, solid_capstyle='round',
                clip_on=False, zorder=10)

        family_angles.setdefault(model, []).append(angle)

    for model, ang_list in family_angles.items():
        mid = np.arctan2(np.mean(np.sin(ang_list)), np.mean(np.cos(ang_list)))
        color = model_colors.get(model, '#AAAAAA')
        ax.text(mid, label_r, model,
                ha='center', va='center', fontsize=fontsize, fontweight='bold',
                color=color, clip_on=False, zorder=11,
                bbox=dict(boxstyle='round,pad=0.2', fc='white',
                          ec=color, lw=1.2, alpha=0.90))


# ─────────────────────────────────────────────
# 3.  IQR BAND
# ─────────────────────────────────────────────

def compute_iqr_band(sub_df, norm_cols, angles):
    """Return closed arrays (q25, q75, mean) aligned with `angles`."""
    q25  = sub_df[norm_cols].quantile(0.25).to_numpy(dtype=float)
    q75  = sub_df[norm_cols].quantile(0.75).to_numpy(dtype=float)
    mean = sub_df[norm_cols].mean(skipna=True).to_numpy(dtype=float)

    def close(arr):
        return np.append(arr, arr[0])

    return close(q25), close(q75), close(mean)


# ─────────────────────────────────────────────
# 4.  SINGLE PROFILE
# ─────────────────────────────────────────────

def plot_single_profile(ax, sub_df, norm_cols, angles, profile_name,
                        color, dim_labels, axis_model, model_colors,
                        individual_alpha=0.10, individual_lw=0.6,
                        mean_lw=2.8, show_annotations=False):
    """
    Draws onto polar axis `ax`:
      - thin individual lines (behavioral heterogeneity)
      - IQR shaded band (25th–75th percentile)
      - thick mean profile line with hollow markers
      - model-family arc segments
    """
    for _, row in sub_df.iterrows():
        v = np.append(row[norm_cols].to_numpy(dtype=float), row[norm_cols].iloc[0])
        ax.plot(angles, v, color=color, alpha=individual_alpha,
                lw=individual_lw, zorder=1)

    q25, q75, mean_vals = compute_iqr_band(sub_df, norm_cols, angles)

    ax.fill_between(angles, q25, q75, color=color, alpha=0.22, zorder=2)
    ax.plot(angles, q25, color=color, alpha=0.45, lw=1.0, ls='--', zorder=4)
    ax.plot(angles, q75, color=color, alpha=0.45, lw=1.0, ls='--', zorder=4)

    ax.plot(angles, mean_vals, 'o-', color=color, lw=mean_lw,
            ms=5, markerfacecolor='white',
            markeredgecolor=color, markeredgewidth=1.8,
            label=f'{profile_name} mean  (n={len(sub_df)})', zorder=5)
    ax.fill(angles, mean_vals, color=color, alpha=0.10, zorder=3)

    draw_model_arcs(ax, angles[:-1], dim_labels, axis_model, model_colors)

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(dim_labels, fontsize=10, color='#333333')
    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.00])
    ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'],
                       fontsize=7.5, alpha=0.5, color='#555555')
    ax.tick_params(axis='x', pad=14)
    ax.spines['polar'].set_color('#CCCCCC')
    ax.grid(color='#DDDDDD', lw=0.6)



# ─────────────────────────────────────────────
# 5.  MAIN FIGURE
# ─────────────────────────────────────────────

def plot_phenotype_profiles_enhanced(
    p1_pop,
    profile_col='profile_p1',
    participant_col='participant_id_canonical',
    save_path=None,
    figsize=(15, 7.5),
):
    """Side-by-side radar: Profile A (left) and Profile B (right)."""
    available  = OrderedDict((k, v) for k, v in RADAR_DIMS.items()
                              if v in p1_pop.columns)
    dim_labels = list(available.keys())
    dim_feats  = list(available.values())
    n_dim      = len(dim_labels)
    am         = {k: AXIS_MODEL[k] for k in dim_labels}

    norm_map = normalize_features(p1_pop, dim_feats)
    p1_radar = p1_pop.copy()
    for feat in dim_feats:
        p1_radar[f'{feat}_norm'] = norm_map[feat]
    norm_cols = [f'{feat}_norm' for feat in dim_feats]

    angles = np.linspace(0, 2 * np.pi, n_dim, endpoint=False)
    angles = np.append(angles, angles[0])

    fig, axes = plt.subplots(1, 2, figsize=figsize, subplot_kw=dict(polar=True))
    fig.subplots_adjust(wspace=0.5)

    specs = [
        ('Profile A', PROFILE_COLORS['Profile A'], axes[0], False),
        ('Profile B', PROFILE_COLORS['Profile B'], axes[1], True),
    ]

    for pname, color, ax, annotate in specs:
        sub = p1_radar[p1_radar[profile_col] == pname].copy()
        if sub.empty:
            ax.set_title(f'{pname}\n(no data)', fontsize=11)
            continue

        plot_single_profile(
            ax=ax, sub_df=sub, norm_cols=norm_cols, angles=angles,
            profile_name=pname, color=color, dim_labels=dim_labels,
            axis_model=am, model_colors=MODEL_COLORS,
            show_annotations=annotate,
        )

        n = sub[participant_col].nunique() if participant_col in sub.columns else len(sub)
        lbl = '(a)' if 'A' in pname else '(b)'
        ax.set_title(f'{lbl}  {pname}\nn = {n} participants',
                     fontsize=11, fontweight='bold', pad=22, color='#222222')
        ax.legend(loc='upper right', bbox_to_anchor=(1.28, 1.15),
                  fontsize=9, framealpha=0.9)

    # Model family legend
    def _desc(m):
        return {'InT': 'Information Theory', 'StoP': 'Stochastic Processes',
                'StM': 'Statistical Mechanics', 'NeT': 'Network Theory'}.get(m, m)

    arc_patches = [mpatches.Patch(color=c, label=f'{m}  —  {_desc(m)}')
                   for m, c in MODEL_COLORS.items()]
    arc_patches.append(mpatches.Patch(facecolor='grey', alpha=0.35,
                                      label='IQR band (25th–75th pct.)'))

    fig.legend(handles=arc_patches, loc='lower center', ncol=3,
               fontsize=9, framealpha=0.95, edgecolor='#CCCCCC',
               bbox_to_anchor=(0.5, -0.06),
               title='Model families contributing each axis', title_fontsize=9.5)

    fig.suptitle(
        'Multidimensional Behavioral Phenotype Profiles\n'
        'Cluster means, IQR bands, and individual participant trajectories',
        fontsize=13, fontweight='bold', y=1.02, color='#111111',
    )

    plt.tight_layout()

    if save_path is not None:
        fig.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f'Saved: {save_path}')

    plt.show()
    return fig, axes


# ─────────────────────────────────────────────
# 6.  CALL
# ─────────────────────────────────────────────

fig, axes = plot_phenotype_profiles_enhanced(
    p1_pop=p1_pop,
    profile_col='profile_p1',
    participant_col='participant_id_canonical',
    save_path=FIG_DIR / 'fig4_phenotype_profiles_enhanced.png',
)


In [ ]:
# G06b.1 — Radar chart: selected profile + underlying individual participants


# ------------------------------------------------------------
# 1. Define phenotype dimensions
# ------------------------------------------------------------

RADAR_DIMS = {
    'Override\ntendency': 'x_mean',
    'Impulsivity\n(noise)': 'sig_hat',
    'Behavioral\nentropy': 'H_emp',
    'Network\nconcentration': 'HHI_net',
    'Habitual\nrepetition': 'rho_self',
    'Temporal\nspread': 'H_hour',
    'Susceptibility': 'chi_behav',
}

# Keep only dimensions that exist in df_full / p1_pop
RADAR_DIMS = {k: v for k, v in RADAR_DIMS.items() if v in df_full.columns}

dim_labels = list(RADAR_DIMS.keys())
dim_feats  = list(RADAR_DIMS.values())
N_dim = len(dim_labels)


# ------------------------------------------------------------
# 2. Normalize each dimension to [0, 1] across the Phase 1 population
# ------------------------------------------------------------

norm_vals = {}

for feat in dim_feats:
    vals = p1_pop[feat]
    vmin, vmax = vals.min(), vals.max()

    norm_vals[feat] = (vals - vmin) / (vmax - vmin + 1e-12)


# Add normalized values directly to p1_pop for easier plotting
p1_radar = p1_pop.copy()

for feat in dim_feats:
    p1_radar[f'{feat}_norm'] = norm_vals[feat]


# ------------------------------------------------------------
# 3. Radar chart setup
# ------------------------------------------------------------

angles = np.linspace(0, 2 * np.pi, N_dim, endpoint=False).tolist()
angles += angles[:1]  # close polygon


# ------------------------------------------------------------
# 4. Function: plot one selected profile with individual participants
# ------------------------------------------------------------

def plot_profile_with_individuals(
    profile_name,
    p1_radar,
    dim_feats,
    dim_labels,
    profile_col='profile_p1',
    participant_col='participant_id_canonical',
    color='#7E57C2',
    individual_alpha=0.18,
    individual_lw=0.8,
    mean_lw=3.0,
    figsize=(7.5, 7.5),
    save_path="radar_individuals"
):
    """
    Radar plot for one selected profile.

    Thin transparent lines:
        individual participants assigned to the selected profile

    Thick line:
        average profile / cluster mean

    Parameters
    ----------
    profile_name : str
        Name of the selected profile, e.g. 'Profile A'.

    p1_radar : pd.DataFrame
        Participant-level dataframe containing profile assignment and normalized features.

    dim_feats : list
        Original feature names used for radar dimensions.

    dim_labels : list
        Labels shown around the radar chart.

    profile_col : str
        Column containing profile labels.

    participant_col : str
        Participant identifier column.

    color : str
        Main color for individual and mean profile lines.

    save_path : str or pathlib.Path, optional
        If provided, saves the figure to this path.
    """

    norm_cols = [f'{feat}_norm' for feat in dim_feats]

    sub = p1_radar[p1_radar[profile_col] == profile_name].copy()

    if sub.empty:
        raise ValueError(f'No participants found for profile: {profile_name}')

    N_dim = len(dim_feats)

    angles = np.linspace(0, 2 * np.pi, N_dim, endpoint=False).tolist()
    angles += angles[:1]

    fig, ax = plt.subplots(
        1,
        1,
        figsize=figsize,
        subplot_kw=dict(polar=True)
    )

    # --------------------------------------------------------
    # Thin individual participant profiles
    # --------------------------------------------------------

    for _, row in sub.iterrows():
        vals = row[norm_cols].to_numpy(dtype=float).tolist()
        vals += vals[:1]

        ax.plot(
            angles,
            vals,
            color=color,
            alpha=individual_alpha,
            linewidth=individual_lw,
            zorder=1
        )

    # --------------------------------------------------------
    # Thick mean profile
    # --------------------------------------------------------

    mean_vals = sub[norm_cols].mean(skipna=True).to_numpy(dtype=float).tolist()
    mean_vals += mean_vals[:1]

    ax.plot(
        angles,
        mean_vals,
        'o-',
        color=color,
        linewidth=mean_lw,
        label=f'{profile_name} mean',
        zorder=5
    )

    ax.fill(
        angles,
        mean_vals,
        color=color,
        alpha=0.12,
        zorder=3
    )

    # --------------------------------------------------------
    # Styling
    # --------------------------------------------------------

    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(dim_labels, fontsize=9)

    ax.set_ylim(0, 1)
    ax.set_yticks([0.25, 0.50, 0.75, 1.00])
    ax.set_yticklabels(['0.25', '0.50', '0.75', '1.00'], fontsize=8, alpha=0.6)

    ax.set_title(
        f'{profile_name}: Individual Participants and Mean Phenotype Profile\n'
        f'n = {sub[participant_col].nunique() if participant_col in sub.columns else len(sub)} participants',
        fontsize=11,
        pad=18
    )

    ax.legend(
        loc='upper right',
        bbox_to_anchor=(1.25, 1.12),
        fontsize=9
    )

    plt.tight_layout()

    if save_path is not None:
        #fig.savefig(FIG_DIR / 'fig4.2_cluster_profiles_radar_individual.png', bbox_inches='tight')
        fig.savefig(save_path, bbox_inches='tight', dpi=300)
        print(f'Saved: {save_path}')

    plt.show()
    return fig, ax

# 2. Plot one selected profile
fig, ax = plot_profile_with_individuals(
    #profile_name='Profile A',  
    profile_name='Profile B',   
    p1_radar=p1_radar,
    dim_feats=dim_feats,
    dim_labels=dim_labels,
    profile_col='profile_p1',
    participant_col='participant_id_canonical',
    color='#7E57C2',
    #save_path=FIG_DIR / 'fig4_profile_A_individual_radar.png'
    save_path=FIG_DIR / 'fig4_profile_B_individual_radar.png'
)

## H — Condition Overlay


In [ ]:
# H01-H02 — Condition distribution per cluster + chi-square test

from scipy.stats import chi2_contingency

ct = pd.crosstab(
    df_full.loc[df_full['cluster_p1'].notna(), 'profile_p1'],
    df_full.loc[df_full['cluster_p1'].notna(), 'study_condition']
)
ct_pct = ct.div(ct.sum(axis=1), axis=0) * 100

print('Condition distribution within each cluster (%):')
print(ct_pct.round(1).to_string())
print(f'\nRaw counts:')
print(ct.to_string())

chi2, p_chi2, dof, expected = chi2_contingency(ct.values)
print(f'\nChi-square test: chi2={chi2:.2f}, df={dof}, p={p_chi2:.4f}')
if p_chi2 < 0.05:
    print('  -> Clusters are NOT equally distributed across conditions (p<0.05)')
    print('     Condition assignment partially predicts cluster membership.')
    print('     Interpret cluster differences carefully — some variance may reflect condition.')
else:
    print('  -> Cluster membership is not significantly associated with condition (p>=0.05)')
    print('     Phenotype profiles cut across experimental conditions.')

# Visualize
fig, ax = plt.subplots(figsize=(9, 4))
n_conds = ct_pct.shape[1]
cond_colors = [plt.cm.Set2(i / max(n_conds - 1, 1)) for i in range(n_conds)]
ct_pct.plot(kind='bar', ax=ax, color=cond_colors, edgecolor='white')
ax.set_title('Condition Distribution by Cluster (Phase 1)', fontsize=11)
ax.set_xlabel('Cluster')
ax.set_ylabel('% within cluster')
ax.legend(title='Condition', bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=9)
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
fig.savefig(FIG_DIR / 'fig5_condition_by_cluster.png', bbox_inches='tight')
plt.close()
print('Saved fig5_condition_by_cluster.png')


## I — Psychosocial Overlays (Post-hoc Only)

**Important:** Psychosocial scales were NOT used to form the clusters (FLAG P4).
All associations below are post-hoc descriptive overlays. They cannot be interpreted
as causal explanations. Bonferroni correction applied across the 5 scales.


In [ ]:
# I01-I04 — ANOVA / Kruskal-Wallis per psychosocial scale + effect sizes

from scipy.stats import kruskal, f_oneway

psych_results = []
bonferroni_n = len(PSYCHOSOCIAL_COLS)

print(f'Psychosocial overlays (Bonferroni-corrected alpha = {0.05/bonferroni_n:.3f})')
print('-' * 80)

for col in PSYCHOSOCIAL_COLS:
    if col not in df_full.columns:
        continue
    groups = [
        df_full.loc[(df_full['profile_p1'] == pname) & df_full[col].notna(), col].values
        for pname in profile_names
    ]
    groups = [g for g in groups if len(g) >= 5]
    if len(groups) < 2:
        continue

    # Kruskal-Wallis (non-parametric)
    h_stat, p_kw = kruskal(*groups)
    p_adj = min(p_kw * bonferroni_n, 1.0)

    # Epsilon-squared effect size
    n_total = sum(len(g) for g in groups)
    eps2 = (h_stat - len(groups) + 1) / (n_total - len(groups))
    eps2 = max(eps2, 0.0)

    sig_flag = '***' if p_adj < 0.001 else ('**' if p_adj < 0.01 else ('*' if p_adj < 0.05 else 'ns'))

    # Per-cluster means
    means_str = '  '.join([f'{pname}={np.mean(g):.2f}' for pname, g in zip(profile_names, groups)])
    print(f'{col:<35s}: H={h_stat:.2f}, p_adj={p_adj:.4f} {sig_flag}, eps2={eps2:.3f}')
    print(f'   Means: {means_str}')

    psych_results.append({
        'scale': col, 'H_stat': h_stat, 'p_kruskal': p_kw,
        'p_bonferroni': p_adj, 'epsilon_squared': eps2,
        **{f'{pn}_mean': np.mean(g) for pn, g in zip(profile_names, groups)},
        **{f'{pn}_n': len(g) for pn, g in zip(profile_names, groups)}
    })

psych_results_df = pd.DataFrame(psych_results)
print(f'\nNote: All psychosocial associations are exploratory and post-hoc.')
print(f'Clusters were formed on behavioral features only (FLAG P4).')


In [ ]:
# I04 — Visualize psychosocial distributions by cluster

available_psych = [c for c in PSYCHOSOCIAL_COLS if c in df_full.columns
                   and df_full[c].notna().sum() > 0]

if available_psych:
    n_plots = len(available_psych)
    fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4), sharey=False)
    if n_plots == 1:
        axes = [axes]

    colors_psych = plt.cm.Set1(np.linspace(0, 0.8, len(profile_names)))

    for ax, col in zip(axes, available_psych):
        data_by_profile = [
            df_full.loc[(df_full['profile_p1'] == pname) & df_full[col].notna(), col].values
            for pname in profile_names
        ]
        bp = ax.boxplot(data_by_profile, patch_artist=True, showfliers=False)
        for patch, color in zip(bp['boxes'], colors_psych):
            patch.set_facecolor(color)
            patch.set_alpha(0.7)
        ax.set_xticklabels(profile_names, fontsize=9)
        ax.set_title(col.replace('_pre', '\n(pre)').replace('_', ' '), fontsize=9)
        ax.set_ylabel('Scale score')
        ax.grid(alpha=0.3)

    fig.suptitle('Psychosocial Scale Distributions by Cluster (post-hoc overlays)', fontsize=11)
    plt.tight_layout()
    fig.savefig(FIG_DIR / 'fig6_psychosocial_by_cluster.png', bbox_inches='tight')
    plt.close()
    print('Saved fig6_psychosocial_by_cluster.png')


## J — Phase 2 Extended Clustering (Sensitivity Feature Set)

Phase 2 adds secondary features to the Phase 1 core set.
The result is compared with Phase 1 using the Adjusted Rand Index (ARI).
Phase 2 is a **sensitivity check**, not a replacement for Phase 1.


In [ ]:
# J01-J04 — Extended clustering with secondary features

# Residualize H_out on condition to remove structural condition confound (FLAG P6)
if 'H_out' in df_p1.columns and 'study_condition' in df_p1.columns:
    # One-hot encode condition
    cond_dummies = pd.get_dummies(df_p1['study_condition'], prefix='cond', drop_first=True)
    from sklearn.linear_model import LinearRegression
    mask_hout = df_p1['H_out'].notna()
    if mask_hout.sum() > 10 and cond_dummies.shape[1] > 0:
        lr = LinearRegression()
        lr.fit(cond_dummies[mask_hout], df_p1.loc[mask_hout, 'H_out'])
        df_p1['H_out_resid'] = np.nan
        df_p1.loc[mask_hout, 'H_out_resid'] = (
            df_p1.loc[mask_hout, 'H_out'].values - lr.predict(cond_dummies[mask_hout])
        )
        print(f'H_out_resid computed (residualized on condition): {df_p1["H_out_resid"].notna().sum()} values')
    else:
        df_p1['H_out_resid'] = df_p1.get('H_out', np.nan)
else:
    df_p1['H_out_resid'] = np.nan 

PHASE2_EXTENDED_FULL = PHASE2_EXTENDED + ['H_out_resid']
PHASE2_EXTENDED_FULL = [f for f in PHASE2_EXTENDED_FULL if f in df_p1.columns]

# Analysis population for Phase 2: complete on Phase 2 features
p2_complete_mask = df_p1[PHASE2_EXTENDED_FULL].notna().all(axis=1)
n_p2 = p2_complete_mask.sum()
print(f'Phase 2 complete-case N: {n_p2} (vs Phase 1 N={len(df_p1)})')
print(f'Phase 2 features used: {PHASE2_EXTENDED_FULL}')

if n_p2 >= 50:
    X_p2 = StandardScaler().fit_transform(df_p1.loc[p2_complete_mask, PHASE2_EXTENDED_FULL].values)

    p2_km_results = {}
    for k in range(2, min(7, n_p2 // 40 + 2)):
        km_p2 = KMeans(n_clusters=k, init='k-means++', n_init=10, random_state=RANDOM_STATE)
        lab_p2 = km_p2.fit_predict(X_p2)
        sil_p2 = silhouette_score(X_p2, lab_p2)
        p2_km_results[k] = {'labels': lab_p2, 'silhouette': sil_p2}
        print(f'Phase 2 k={k}: silhouette={sil_p2:.3f}, counts={pd.Series(lab_p2).value_counts().sort_index().values.tolist()}')

    # Select Phase 2 k using the SAME eligibility rule as Phase 1 (cell E06-E08):
    # silhouette > 0.2 AND min cluster size >= 40.
    p2_eligible = [k for k, r in p2_km_results.items()
                   if r['silhouette'] > 0.2 and pd.Series(r['labels']).value_counts().min() >= 40]
    if p2_eligible:
        P2_K = max(p2_eligible, key=lambda k: p2_km_results[k]['silhouette'])
    else:
        P2_K = max(p2_km_results, key=lambda k: p2_km_results[k]['silhouette'])
        print('WARNING: no Phase 2 k meets the Phase 1 eligibility criteria; '
              'selecting by silhouette alone.')
    print(f'Eligible Phase 2 k (sil>0.2, min_size>=40): {p2_eligible}')
    print(f'Selected Phase 2 k = {P2_K}  '
          f"(silhouette {p2_km_results[P2_K]['silhouette']:.4f}; "
          f"runner-up margin over k=2: {p2_km_results[P2_K]['silhouette'] - p2_km_results[2]['silhouette']:.4f})")
    # Same-k reference point: ARI is systematically depressed across different k, so also
    # report the k=2-vs-k=2 comparison alongside the selected-k one.
    ari_p1_p2_samek = adjusted_rand_score(df_p1.loc[p2_complete_mask, 'cluster_p1'].values,
                                          p2_km_results[2]['labels'])
    print(f'ARI (Phase 1 k=2 vs Phase 2 k=2, same-k reference): {ari_p1_p2_samek:.3f}')

    # Compare with Phase 1 on the overlap population
    p1_labels_sub = df_p1.loc[p2_complete_mask, 'cluster_p1'].values
    p2_labels_final = p2_km_results[P2_K]['labels']
    ari_p1_p2 = adjusted_rand_score(p1_labels_sub, p2_labels_final)
    print(f'\nARI (Phase 1 k=2 vs Phase 2 k={P2_K}, overlap population): {ari_p1_p2:.3f}'
          '  [cross-k ARI — not on the same scale as the same-k checks above]')
    if ari_p1_p2 > 0.70:
        print('  -> High ARI: secondary features do NOT substantially change cluster structure')
        print('     Phase 1 solution is robust to addition of secondary features.')
    elif ari_p1_p2 > 0.50:
        print('  -> Moderate ARI: secondary features partially reorganize clusters')
    else:
        print('  -> Low ARI: secondary features substantially change cluster structure')
        print('     Report Phase 2 results alongside Phase 1 — the extended feature set captures additional variance.')
    df_p1['cluster_p2'] = np.nan
    df_p1.loc[p2_complete_mask, 'cluster_p2'] = p2_labels_final
else:
    print(f'WARNING: Phase 2 N={n_p2} too small for clustering. Skipping Phase 2.')
    P2_K = None
    ari_p1_p2 = None
    df_p1['cluster_p2'] = np.nan


## K — Phenotype Descriptions


In [ ]:
# K01-K03 — Generate cautious descriptive profile for each cluster
#
# Language follows blueprint §7.1:
#   - Descriptive behavioral profiles, NOT diagnostic types
#   - Paper §4.3 dimension mapping
#   - No intervention prescription (Step 3 territory)

print('=' * 70)
print('PHASE 1 CLUSTER PROFILES — DESCRIPTIVE BEHAVIORAL PROFILES')
print('=' * 70)
print()
print('Note: All descriptions are behavioral profiles observed during the DS5 study period')
print('(7-45 days). They describe patterns in proxy measurements (X_t = opens/events),')
print('NOT stable personality traits or latent psychological dispositions.')
print()

# Compute profile summaries
for pname in profile_names:
    sub = df_full[df_full['profile_p1'] == pname]
    n_prof = len(sub)
    print(f'--- {pname} (N={n_prof}) ---')

    for feat in PHASE1_CORE:
        if feat in sub.columns:
            v = sub[feat].dropna()
            print(f'  {feat:<15s}: mean={v.mean():.3f}, median={v.median():.3f}, std={v.std():.3f}')

    # Interpretation overlay
    for flag_col in ['is_dys_eligible', 'behaviorally_deterministic', 'four_model_convergent', 'high_chi']:
        if flag_col in sub.columns:
            vals = sub[flag_col].dropna().astype(float)
            if len(vals) > 0:
                print(f'  {flag_col:<15s}: {100*vals.mean():.1f}% ({len(vals)} with data)')

    # Condition
    cond_dist = sub['study_condition'].value_counts(normalize=True) * 100
    cond_str = ', '.join([f'{c}: {pct:.0f}%' for c, pct in cond_dist.items()])
    print(f'  Condition mix: {cond_str}')

    # Paper dimensions narrative
    sig = sub['sig_hat'].dropna().mean() if 'sig_hat' in sub.columns else np.nan
    xm  = sub['x_mean'].dropna().mean()  if 'x_mean' in sub.columns else np.nan
    hhi = sub['HHI_net'].dropna().mean() if 'HHI_net' in sub.columns else np.nan
    rho = sub['rho_self'].dropna().mean() if 'rho_self' in sub.columns else np.nan

    # Compare like with like: cluster MEAN vs population MEAN. Both features are right-skewed
    # (sig_hat mean 0.206 / median 0.181; x_mean mean 0.694 / median 0.740), so thresholding a
    # cluster mean against the population median put BOTH clusters on the same side of both
    # features and produced identical, non-discriminating descriptions.
    sig_label = 'high variability (noise-dominated)' if sig > df_p1['sig_hat'].mean() else 'low variability (habit-stable)'
    xm_label  = 'high override tendency' if xm > df_p1['x_mean'].mean() else 'low override tendency'

    print(f'  Behavioral description: {sig_label}; {xm_label};')
    print(f'    Network concentration (HHI): {hhi:.3f}; Habitual repetition (rho_self): {rho:.3f}')
    print()

print('Reminder: do NOT assign intervention prescriptions to these profiles (Step 3 territory).')


## L — Limitations

**L01 — Mandatory limitations (from blueprint §7.5 and §13)**

1. **Short window caveat.** DS5 observation windows are 7–45 days. Cluster assignments
   reflect behavioral profiles during the study period, not stable individual traits.
   Cross-dataset replication (DS4 naturalistic, DS6 repeated waves) is required before
   claiming stable phenotypes.

2. **Proxy limitation.** All features derive from X_t = n_opened / n_events. This behavioral
   proxy does not directly measure urge, self-control, habit strength, or any latent
   psychological state. Phenotype labels describe patterns in the proxy, not psychological types.

3. **NeT metrics near ceiling.** HHI_net ≈ 0.84–0.86 and rho_self ≈ 0.92–0.92 across all
   DS5 participants. Both variables are compressed near their upper bounds; individual variation
   is subtle and cluster differences on these dimensions should be interpreted cautiously.

4. **H_out condition confound.** Friction outcome entropy (H_out) is partially driven by
   experimental condition (control participants cannot produce the same mix of dismiss/close
   outcomes). H_out was excluded from Phase 1 clustering (FLAG P6); residualized H_out was
   used in Phase 2 only.

5. **Effective dimensionality is lower than it appears.** Within the StoP↔InT↔DyS↔StM
   cluster, only two genuinely independent axes exist (sig_hat, H_emp). NeT adds 1–2 more.
   The 5-feature Phase 1 space effectively spans 3–4 independent behavioral dimensions.

6. **Step 2 is discovery, not confirmation.** Cluster structures are hypotheses.
   They require replication in DS4 and DS6 before reporting as robust phenotypes.


## M — Forward Connections (Step 3 Setup)

**M01 — What these profiles imply for deferred models (not prescriptive)**

- **Game Theory (GaT, notebook 46):** GaT requires player types to parameterize
  payoff matrices. The Step 2 profiles provide behavioral descriptions (high-override,
  low-override, variable vs. stable) that can be mapped to strategic agent categories
  (e.g., stable self-regulators, chronic over-riders, adaptive learners). GaT implementation
  can now proceed using profile membership as agent initialization.

- **Agent-Based Modeling (ABM, notebook 47):** ABM agent parameters (sig_hat as noise;
  x_mean as baseline tendency; chi_behav as response sensitivity) are now available per
  cluster from Step 1 outputs. Step 2 cluster profiles define discrete agent types.
  ABM can initialize distinct agent populations using cluster-level parameter estimates.

- **System Dynamics (SyD, notebook 48):** SyD requires repeated within-person survey
  waves. DS5 has only pre/post — insufficient. Deferred to DS6.

**M02 — Step 3 signal question**

Step 3 will ask: *Which cluster (phenotype profile) benefits most from which friction type?*
DS5 is the correct first Step 3 dataset because condition assignment was experimental
(randomized). This question is NOT answered here — it is flagged for Step 3 work.

**Do not add any intervention prescription logic to this notebook.**


## N — Save Outputs


In [ ]:
# N01 — Save ds5_phenotype_feature_matrix.csv

feat_matrix_cols = (
    ['participant_id_canonical', 'study_condition']
    + PHASE1_CORE
    + [f for f in PHASE2_SENSITIVITY_EXTRA if f in df_p1.columns]
    + ['H_out', 'H_SDE', 'mu_hat', 'sigma_mu_ratio',
       'is_dys_eligible', 'behaviorally_deterministic', 'four_model_convergent',
       'beta_behav', 'chi_behav', 'F_behav', 'H_trans', 'I_consec',
       'H_out_resid']
)
feat_matrix_cols = [c for c in feat_matrix_cols if c in df_p1.columns]
feat_matrix = df_p1[feat_matrix_cols].copy()
feat_matrix.to_csv(OUTPUT_DIR / 'ds5_phenotype_feature_matrix.csv', index=False)
print(f'Saved ds5_phenotype_feature_matrix.csv ({feat_matrix.shape[0]} rows x {feat_matrix.shape[1]} cols)')


In [ ]:
# N02 — Save ds5_feature_redundancy_report.csv

red_report = pd.DataFrame([
    dict(feature='sig_hat',    role='CORE_CLUSTERING', rationale='StoP backbone; independent impulsivity/noise axis'),
    dict(feature='H_emp',      role='CORE_CLUSTERING', rationale='InT backbone; partially independent of sig_hat (r~0.65); non-Gaussian patterns'),
    dict(feature='x_mean',     role='CORE_CLUSTERING', rationale='Mean override level; independent; compliance dimension'),
    dict(feature='HHI_net',    role='CORE_CLUSTERING', rationale='NeT backbone; genuinely new; network hub-concentration'),
    dict(feature='rho_self',   role='CORE_CLUSTERING', rationale='NeT; genuinely new; same-app habitual repetition'),
    dict(feature='H_hour',     role='SECONDARY',       rationale='InT; independent; temporal diversity; context-dependence'),
    dict(feature='H_app',      role='SECONDARY',       rationale='InT; independent; app breadth; correlated with H_trans'),
    dict(feature='chi_behav',  role='SECONDARY',       rationale='StM; partial new signal via p_use; susceptibility near tipping point'),
    dict(feature='H_out_resid',role='SECONDARY(P2)',   rationale='Residualized H_out; partial new signal after condition adjustment'),
    dict(feature='mu_hat',     role='SECONDARY',       rationale='StoP; near-zero across DS5; weak discriminator; include in sensitivity only'),
    dict(feature='H_trans',    role='SECONDARY',       rationale='NeT; partially new; correlated with H_app via I_consec'),
    dict(feature='beta_behav', role='INTERPRETATION_ONLY', rationale='= 2/sig_hat^2; monotone transform; no new clustering information'),
    dict(feature='T_behav',    role='INTERPRETATION_ONLY', rationale='= sig_hat^2/2; monotone transform'),
    dict(feature='H_SDE',      role='INTERPRETATION_ONLY', rationale='= f(sig_hat); monotone transform'),
    dict(feature='F_behav',    role='INTERPRETATION_ONLY', rationale='= f(sig_hat, H_emp); composite of two core features'),
    dict(feature='is_dys_eligible', role='INTERPRETATION_ONLY', rationale='Binarization of sig_hat'),
    dict(feature='behaviorally_deterministic', role='INTERPRETATION_ONLY', rationale='Composite of low sig_hat AND low H_emp'),
    dict(feature='four_model_convergent', role='INTERPRETATION_ONLY', rationale='Effectively = behaviorally_deterministic'),
    dict(feature='I_consec',   role='EXCLUDE', rationale='= H_app - H_trans; derived; not independent of H_app and H_trans'),
    dict(feature='top1_share_net', role='EXCLUDE', rationale='Correlated with HHI_net'),
    dict(feature='sigma_mu_ratio', role='EXCLUDE', rationale='31.2% NaN; too noisy for clustering'),
    dict(feature='p_use_approx', role='EXCLUDE', rationale='Near-duplicate of x_mean (r>0.95)'),
    dict(feature='H_pi',       role='EXCLUDE(P1)', rationale='Lower coverage (66.4%); bring in if Phase 1 stable'),
    dict(feature='study_condition', role='OVERLAY_ONLY', rationale='Experimental assignment; NOT a behavioral feature'),
])
red_report.to_csv(OUTPUT_DIR / 'ds5_feature_redundancy_report.csv', index=False)
print(f'Saved ds5_feature_redundancy_report.csv ({len(red_report)} features classified)')


In [ ]:
# N03 — Save ds5_cluster_assignments.csv

assign_df = df_p1[['participant_id_canonical', 'study_condition', 'cluster_p1', 'profile_p1']].copy()
if 'cluster_p2' in df_p1.columns:
    assign_df['cluster_p2'] = df_p1['cluster_p2'].values
assign_df.to_csv(OUTPUT_DIR / 'ds5_cluster_assignments.csv', index=False)
print(f'Saved ds5_cluster_assignments.csv ({len(assign_df)} rows)')


In [ ]:
# N04 — Save ds5_cluster_profiles.csv

profile_df.to_csv(OUTPUT_DIR / 'ds5_cluster_profiles.csv', index=False)
print(f'Saved ds5_cluster_profiles.csv ({len(profile_df)} rows x {len(profile_df.columns)} cols)')


In [ ]:
# N05 — Save ds5_psychosocial_overlays.csv

if len(psych_results) > 0:
    psych_results_df.to_csv(OUTPUT_DIR / 'ds5_psychosocial_overlays.csv', index=False)
    print(f'Saved ds5_psychosocial_overlays.csv ({len(psych_results_df)} scales)')
else:
    print('No psychosocial results to save (check column availability).')


In [ ]:
# N06 — Write model_run_summary.md

import datetime

n_p1 = len(df_p1)
sil_selected = km_results[SELECTED_K]['silhouette']
jac_str = f'{jac_mean:.3f} (std={jac_std:.3f})'

summary_lines = [
    '# DS5 Step 2 — Phenotyping Model Run Summary\n',
    f'\n**Date:** {datetime.date.today()}\n',
    f'**Notebook:** 100_phenotype_exploration.ipynb\n',
    f'**Stage:** Stage 4, Step 2\n',
    '\n---\n',
    '\n## Run parameters\n',
    f'- Phase 1 features: {PHASE1_CORE}\n',
    f'- Phase 2 sensitivity extras: {PHASE2_SENSITIVITY_EXTRA}\n',
    f'- Phase 1 analysis population (complete case): N={n_p1}\n',
    f'- Algorithm: K-means (k-means++ init, 10 seeds)\n',
    f'- Selected k: {SELECTED_K}\n',
    f'- RANDOM_STATE: {RANDOM_STATE}\n',
    '\n## Cluster selection metrics\n',
]

for k in list(K_RANGE):
    r = km_results[k]
    summary_lines.append(
        f'- k={k}: silhouette={r["silhouette"]:.3f}, CH={r["CH"]:.1f}, '
        f'DB={r["DB"]:.3f}, min_size={r["min_size"]}\n'
    )

summary_lines += [
    f'\n**Selected solution:** k={SELECTED_K}, silhouette={sil_selected:.3f}\n',
    f'\n## Robustness\n',
    f'- Bootstrap Jaccard stability (100 bootstraps, 80% sample): {jac_str}\n',
    f'- Stability verdict: {stability_verdict}\n',
    f'- ARI (drop H_emp sensitivity): {ari_sens:.3f}\n',
]

if ari_p1_p2 is not None:
    summary_lines.append(f'- ARI (Phase 1 vs Phase 2 extended): {ari_p1_p2:.3f}\n')

summary_lines += ['\n## Cluster profiles (core features)\n']
for pname in profile_names:
    sub = df_full[df_full['profile_p1'] == pname]
    summary_lines.append(f'\n### {pname} (N={len(sub)})\n')
    for feat in PHASE1_CORE:
        if feat in sub.columns:
            v = sub[feat].dropna()
            summary_lines.append(f'- {feat}: mean={v.mean():.3f}, median={v.median():.3f}\n')

summary_lines += [
    '\n## Psychosocial overlays\n',
    '(Bonferroni-corrected p threshold = 0.010 for 5 scales)\n',
]
if len(psych_results) > 0:
    for r in psych_results:
        flag = '(significant)' if r['p_bonferroni'] < 0.05 else '(ns)'
        summary_lines.append(f"- {r['scale']}: H={r['H_stat']:.2f}, p_adj={r['p_bonferroni']:.4f} {flag}\n")

summary_lines += [
    '\n## Key limitations\n',
    '1. Short observation windows (7-45 days): profiles may reflect study-period behavior, not stable traits\n',
    '2. Proxy: all features from X_t = n_opened/n_events\n',
    '3. NeT metrics near ceiling (HHI ~0.84, rho_self ~0.92): subtle individual variation\n',
    '4. H_out condition-confounded (excluded from Phase 1)\n',
    '5. Effective dimensionality ~3-4 despite 5 input features\n',
    '6. Step 2 = discovery; cross-dataset replication required\n',
    '\n## Output files\n',
    '- ds5_phenotype_feature_matrix.csv\n',
    '- ds5_feature_redundancy_report.csv\n',
    '- ds5_cluster_assignments.csv\n',
    '- ds5_cluster_profiles.csv\n',
    '- ds5_psychosocial_overlays.csv\n',
    '- figures/fig0_univariate_distributions.png\n',
    '- figures/fig1_feature_correlation_matrix.png\n',
    '- figures/fig2_pca_biplot.png\n',
    '- figures/fig3_cluster_selection_metrics.png\n',
    '- figures/fig4_cluster_profiles_radar.png\n',
    '- figures/fig5_condition_by_cluster.png\n',
    '- figures/fig6_psychosocial_by_cluster.png\n',
    '- figures/fig7_dendrogram.png\n',
]

with open(OUTPUT_DIR / 'model_run_summary.md', 'w') as f:
    f.writelines(summary_lines)
print('Saved model_run_summary.md')
print('\nAll outputs saved to:', OUTPUT_DIR)
